# Lake-level perturbations at lake-terminating glaciers (ISSM)

The experiments of Bézu, Nick and Immerzeel (in revision for JGR: Earth Surface), in ISSM: an idealized glacier and Luggye Glacier, each as a flowline and in plan view. Each glacier is brought to a reference state, the rate and position laws are calibrated to it, and the lake level is then ramped over one year to offsets from −10 to +10 m under each calving law. The icepack version, `lake_terminating_GRL_icepack.ipynb`, has the same structure and choices for the flowlines.

**Sliding laws.** Power law $\tau_b = C|u_b|^{1/m}$; effective pressure $\tau_b = C f_{af}|u_b|^{1/m}$, with the fraction above flotation $f_{af} = (b - z_w + \varrho H)/H$ clamped to [0.01, 1] ($b$ the ice base, $z_w$ the lake level, $\varrho = \rho_i/\rho_w$).

**Calving laws**, for the terminus position $L$: rate law $\dot L = u_t - k\,d_{gl}$; position law $L = \max\{x : f_{af} \geq f^*\}$, reached by advancing with the flow; trivial law (a fixed front) $\dot L = 0$. Here $u_t$ is the terminus speed and $d_{gl}$ the water depth at the grounding line. $k$ and $f^*$ are calibrated so that, with the lake level unchanged, the terminus ends the year where it started (within 0.5 m), starting from $k = u_t/d_{gl}$ and $f^* = f_{af}$ at the terminus of the reference glacier.

**Layout.** Four experiments: the idealized glacier as a flowline, and in plan view with two kinds of valley walls; Luggye as a flowline, and in plan view. Each computes and saves its results and prints a summary. The **Figures** at the end are drawn from those results alone, and **Notes on the method** at the very end hold the details. The plan-view parts need the current `issm_helper` (with `valley_mesh`, `front_diagnostics` and the interpolated front in `diagnostic_solve`).

**Progress** is saved after every run (and in the plan view, every 10 model years of a spin-up and after every calibration try), to a file per part named after its parameters (e.g. `GRL_idealized_3f9a1c2e.npy`). If the kernel stops, rerun the notebook to continue where it left off; with all runs saved, a rerun takes minutes. Changing a parameter starts a new file, and changing it back picks up the earlier results. Edits to the code are not detected: delete the files after changing the code.

## Imports

In [1]:
import os
import io
import sys
import copy
import contextlib
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import tqdm
from IPython.display import display, Pretty
import geopandas as gpd
import rasterio
import scipy.ndimage
from shapely.ops import substring
from pyproj import Transformer
from types import SimpleNamespace
from matplotlib.tri import Triangulation, LinearTriInterpolator

from model import model
from issm_helper import *
import issm_helper

## Shared parameters

`n_proc` is the number of MPI processes for the plan-view models, at most the cores this session may use (if `mpiexec` cannot start them, `issm_helper` warns and carries on with one). The lake is fresh water, at 0 m in the reference state. The ice's rate factor is icepack's, as in the icepack version: it replaces the helper's Cuffey & Paterson (2010) and makes the ice 2.1 times softer at 0 °C. Every model has `vertical_layers` evenly spaced layers (see *Notes*). `speed_distance` is how far behind the initial terminus the response figures take the speed.

In [2]:
T = 273.15 #ice temperature (K)

def icepack_rate_factor(T): #icepack's rate factor (Pa^-3 s^-1)
    T = np.asarray(T, dtype = float)
    A_0 = np.where(T < 263.15, 3.985e-13, 1.916e3) #rate prefactor (s^-1 Pa^-3)
    Q = np.where(T < 263.15, 60e3, 139e3) #activation energy (J mol^-1)
    return A_0*np.exp(-Q/(8.3144621*T))
issm_helper.rate_factor = icepack_rate_factor

m = 3 #Weertman exponent
ρ_w = 1000 #lake water density (kg/m³)
ϱ = model().materials.rho_ice/ρ_w
yts = model().constants.yts

sliding_laws = ['power-law', 'effective-pressure']
calving_laws = ['rate law', 'position law', 'trivial law']
offsets = np.arange(-10, 10 + 1) #lake-level perturbations (m)
ramp_time = 1 #each offset is reached by a linear ramp over one year (a)
vertical_layers = 10 #evenly spaced layers in every model
n_proc = min(8, len(os.sched_getaffinity(0))) #MPI processes for the plan-view models (at most the cores this session may use)
speed_distance = 100 #the response figures take the speed this far behind the initial front (m): at that point on a flowline, and near the centreline or flowline in plan view

## Shared functions

- `friction`: the basal friction coefficient for either sliding law. ISSM's drag goes as $C^2$, so effective-pressure sliding passes $C\sqrt{f_{af}}$, evaluated from the current geometry and lake level. `depth_average`: speeds are depth-averaged over the (evenly spaced) layers.
- `progress`: progress bars (`tqdm`) that redraw in place in the notebook, also when papermill runs it, and show live in `progress_log` (the job's log under papermill).
- `progress_file`: a part's progress file, its contents so far, and a function that saves it.
- `calibrate`: the value of $k$ or $f^*$ for which the unperturbed run's terminus drift is within 0.5 m of zero, in at most 12 tries; each try goes to `progress_log`, and the notebook keeps the result (see *Notes*).
- `perturbation_run` (flowline) and `perturbation_run_2d` (plan view): one year with the lake level ramped to an offset under a calving law. The flowline run returns the terminus position, the terminus speed and the grounding-line flux; the plan-view run, the mean length (ice area per unit `width`), the speed at `points` and the flux through the front per unit width (see *Notes*).
- `points_behind` and `speed_at`: the points `speed_distance` behind the initial front on lines down the glacier, and the mean speed there.
- `flowline_profile`: a flowline glacier's bed, ice and speed, for Figure 1.

In [3]:
def friction(md, sliding_law, C):
    if sliding_law == 'power-law':
        return C
    H = md.geometry.thickness
    f_af = np.clip((md.geometry.base - md.initialization.sealevel + ϱ*H)/H, 0.01, 1)
    return C*np.sqrt(f_af)

def depth_average(md, field):
    f = np.reshape(field, (md.mesh.numberoflayers, -1))
    return (f[1:] + f[:-1]).mean(axis = 0)/2

def perturbation_run(md, sliding_law, C, calving_law, offset, k, f_star, Δt):
    x = md.mesh.x[:md.mesh.numberofvertices2d]
    H = md.geometry.thickness[:md.mesh.numberofvertices2d]
    L = front = terminus_position(md) #terminus position, and the node the ISSM front sits on (m)

    for step in range(int(round(ramp_time/Δt))):
        waterline = offset*(step + 1)*Δt/ramp_time
        md.initialization.sealevel = waterline*np.ones(md.mesh.numberofvertices)
        set_thickness(md, H)
        md.mask.ice_levelset = md.mesh.x - front

        u = diagnostic_solve(md, friction = friction(md, sliding_law, C))
        H, s = prognostic_solve(md, Δt)

        x_p, ū = profile(md, depth_average(md, u[0]))
        base = profile(md, md.geometry.base)[1]
        thickness = profile(md, md.geometry.thickness)[1]
        x_gl = np.fmin(grounding_line_position(md), front)
        u_t = np.interp(front, x_p, ū)
        d_gl = waterline - np.interp(x_gl, x_p, base)

        if calving_law == 'rate law':
            L += (u_t - k*d_gl)*Δt
        if calving_law == 'position law':
            L += u_t*Δt #advance with the flow, then calve back to where the ice is f* above flotation
            ice = x_p <= front
            f_af = ((base - waterline + ϱ*thickness)/thickness)[ice]
            if f_af[-1] < f_star:
                j = np.flatnonzero(f_af >= f_star)[-1]
                x_i = x_p[ice]
                L = min(L, x_i[j] + (f_af[j] - f_star)/(f_af[j] - f_af[j + 1])*(x_i[j + 1] - x_i[j]))

        new_front = x_p[np.argmin(np.abs(x_p - L))]
        H = np.where((x > front) & (x <= new_front), np.interp(front, x_p, thickness), H)
        front = new_front

    set_thickness(md, H) #the final state, with the front on its final node and the lake at its final level
    md.mask.ice_levelset = md.mesh.x - front
    u = diagnostic_solve(md, friction = friction(md, sliding_law, C))
    x_p, ū = profile(md, depth_average(md, u[0]))
    thickness = profile(md, md.geometry.thickness)[1]
    x_gl = np.fmin(grounding_line_position(md), front)
    return L, np.interp(front, x_p, ū), np.interp(x_gl, x_p, ū)*np.interp(x_gl, x_p, thickness)

try: #the kernel's own stderr, which the notebook does not capture: the job's log when papermill runs the notebook (the server's console in Jupyter)
    progress_log = open(sys.stderr.fileno(), 'w', buffering = 1, closefd = False)
except (OSError, io.UnsupportedOperation):
    progress_log = open(os.devnull, 'w')

class progress(tqdm.tqdm): #a tqdm bar that redraws in place in the notebook, also when papermill runs it (one output, in its last state), and live in progress_log
    def __init__(self, *args, **kwargs):
        self._handle = None
        super().__init__(*args, file = progress_log, **kwargs)
    def display(self, msg = None, pos = None):
        super().display(msg, pos)
        bar = Pretty(self.__str__() if msg is None else msg)
        if self._handle is None:
            self._handle = display(bar, display_id = True)
        else:
            self._handle.update(bar)
        return True

def progress_file(prefix, settings): #a part's progress file, named after everything its results depend on: its name, what it holds so far, and a function that saves it
    name = f'{prefix}_{hashlib.md5(str(settings).encode()).hexdigest()[:8]}.npy'
    saved = np.load(name, allow_pickle = True).item() if os.path.exists(name) else {}
    print('progress file:', name)
    return name, saved, lambda: np.save(name, saved)

def flowline_profile(md, u): #a flowline glacier for the figures: bed, ice base and surface, and the depth-averaged speed, along x
    x_p, bed = profile(md, md.geometry.bed)
    return {'x': x_p, 'bed': bed, 'base': profile(md, md.geometry.base)[1], 'surface': profile(md, md.geometry.surface)[1],
            'speed': profile(md, depth_average(md, u[0]))[1]}

def perturbation_run_2d(new_model, sliding_law, C, calving_law, offset, k, f_star, Δt, width, points):
    md = new_model(calving_law)
    n2 = md.mesh.numberofvertices2d
    bed = md.geometry.bed[:n2]

    for step in range(int(round(ramp_time/Δt))):
        waterline = offset*(step + 1)*Δt/ramp_time
        md.initialization.sealevel = waterline*np.ones(md.mesh.numberofvertices)
        set_thickness(md, md.geometry.thickness) #the geometry in hydrostatic equilibrium with the new lake level
        diagnostic_solve(md, friction = friction(md, sliding_law, C))
        if calving_law == 'rate law':
            rate_law(md, k*np.maximum(waterline - bed, 0)) #at k times the local water depth
        if calving_law == 'position law':
            H, base = md.geometry.thickness[:n2], md.geometry.base[:n2]
            position_law(md, forcing = -(base - waterline + ϱ*H)/H, threshold = -f_star) #calve where f_af < f*
        prognostic_solve(md, Δt)

    set_thickness(md, md.geometry.thickness)
    u = diagnostic_solve(md, friction = friction(md, sliding_law, C)) #the final state
    front = front_diagnostics(md)
    return front.area/width, speed_at(md, u, points), front.flux/width

def points_behind(x, y, triangles, levelset, paths, distance): #the points on each path (m) distance (m) upstream of where it first leaves the ice of levelset
    φ_at = LinearTriInterpolator(Triangulation(x, y, np.asarray(triangles, dtype = int)), levelset)
    points = []
    for path in paths:
        φ = φ_at(*path.T).filled(np.nan)
        i = np.flatnonzero((φ[:-1] <= 0) & (φ[1:] > 0))
        if i.size:
            i = i[0]
            arc = np.r_[0, np.cumsum(np.hypot(*np.diff(path, axis = 0).T))] #distance along the path
            at_front = arc[i] + (arc[i + 1] - arc[i])*(0 - φ[i])/(φ[i + 1] - φ[i])
            points.append((np.interp(at_front - distance, arc, path[:, 0]), np.interp(at_front - distance, arc, path[:, 1])))
    return np.array(points)

def speed_at(md, u, points): #mean depth-averaged speed (m/a) at points
    n2 = md.mesh.numberofvertices2d
    speed = LinearTriInterpolator(Triangulation(md.mesh.x[:n2], md.mesh.y[:n2], md.mesh.elements2d - 1),
                                  np.hypot(depth_average(md, u[0]), depth_average(md, u[1])))
    return float(np.mean(speed(*points.T)))

def remembered(drift, tries, save): #drift, keeping each try in tries and saving them, so that a restarted calibration replays its tries
    def known_drift(value):
        if value not in tries:
            tries[value] = drift(value)
            save()
        return tries[value]
    return known_drift

def calibrate(drift, guess, step, name):
    tries = {}
    value, move, sides = guess, step, []
    for attempt in range(12):
        tries[value] = drift(value)
        print(f'{name} = {value:.4f}: terminus drift {tries[value]:+.2f} m', file = progress_log)
        if abs(tries[value]) < 0.5:
            break
        advanced = [v for v in tries if tries[v] > 0] #too little calving: the terminus advanced
        retreated = [v for v in tries if tries[v] < 0] #too much: it retreated
        if advanced and retreated: #zero drift lies between the closest tries on either side
            low, high = max(advanced), min(retreated)
            if high - low < step/50:
                break
            sides.append(np.sign(tries[value]))
            d_low = tries[low]/2 if sides[-2:] == [-1, -1] else tries[low] #a side that survived the last two tries counts half,
            d_high = tries[high]/2 if sides[-2:] == [1, 1] else tries[high] #so the next try moves away from it
            value = low + (high - low)*d_low/(d_low - d_high) #where the straight line between them reaches zero drift
        else: #not there yet: follow the straight line through the last two tries, at most four times as far as the last move
            if len(tries) > 1:
                previous = list(tries)[-2]
                line = abs(tries[value]*(value - previous)/(tries[value] - tries[previous])) if tries[value] != tries[previous] else np.inf
                move = min(line, 4*move)
            value = value + move*np.sign(tries[value]) #more calving if the terminus advanced, less if it retreated
    best = min(tries, key = lambda v: abs(tries[v]))
    return best, tries[best]

# Idealized glacier (flowline)

## Parameters

A 5 km glacier on a 5% prograde bed, in a lake 50 m deep. $C$ is icepack's 0.2 MPa (m/a)$^{-1/3}$ in ISSM's convention, $\tau_b = C^2|u_b|^{1/m}$.

In [4]:
L_0 = 5000 #initial glacier length (m)
H_0 = 200 #initial ice thickness (m)
slope = 0.05 #mean bed slope, prograde
d_0 = 50 #lake depth at the terminus (m)
D_b, L_b = 50, 1000 #depth and length of the trough behind the retrograde terminus (m)
C = np.sqrt(0.2e6*yts**(1/m)) #Weertman C
u_in = 10 #inflow speed at the head (m/a)

directions = ['retrograde', 'prograde'] #bed slope just upstream of the terminus
Δt_spinup = 0.5 #spin-up time step (a)
max_spinup = 400 #longest spin-up (a)
Δt = 0.1 #perturbation time step (a)

## Geometry

The prograde bed is a plane sloping down to the lake. The retrograde bed is the same plane with a smooth trough just behind the terminus: one cosine dip, `L_b` long and `D_b` deep at its centre (relative to the plane), ending at the terminus, so the lake depth at the terminus is unchanged and the two directions differ only upstream. The surface starts smooth, so the ice is thicker over the trough. Beyond the terminus the lake floor stays level. Nodes are 1 m apart where the front moves, and 50 m elsewhere.

In [5]:
x_nodes = np.unique(np.r_[np.arange(0, 4800, 50), np.arange(4800, 4880, 5), np.arange(4880, 5040, 1),
                          np.arange(5040, 5100, 5), np.arange(5100, 7000 + 1, 50)]) #node positions, with 2 km of lake beyond the glacier (m)
md = flowline_mesh(model(), x_nodes)
x = md.mesh.x

s_t = np.maximum(L_0 - x, 0) #distance upstream of the terminus (m)
plane = -d_0 + slope*s_t #the prograde bed
trough = np.where(s_t < L_b, D_b*(1 - np.cos(2*np.pi*s_t/L_b))/2, 0) #the cosine dip behind the retrograde terminus
beds = {'retrograde': plane - trough, 'prograde': plane}
thicknesses = {direction: plane + H_0 - bed for direction, bed in beds.items()}

params = {'approximation': 'HO',
          'vertical_layers': vertical_layers,
          'temperature': T,
          'friction': C, #replaced by friction(md, sliding_law, C) at every solve
          'm': m,
          'rho_water': ρ_w,
          'ice_levelset': x - L_0,
          'spcvx': dirichlet(md, (x == 0, u_in)),
          'spcthickness': dirichlet(md, (x == 0, H_0))}

## Spin-up and calibration

The trivial law (fixed front), until the ice volume changes by less than $10^{-5}$ per year, for at most `max_spinup` years. The terminus is grounded, so $d_{gl}$ is the water depth at the terminus. Then $k$ and $f^*$ are calibrated for zero terminus drift.

In [6]:
settings = {'calibration': 'zero drift', 'flux': 'at the final position', 'vertical_layers': vertical_layers, 'T': T, 'A': float(issm_helper.rate_factor(T)), 'm': m, 'ρ_w': ρ_w, 'offsets': offsets.tolist(), 'ramp_time': ramp_time,
            'L_0': L_0, 'H_0': H_0, 'slope': slope, 'd_0': d_0, 'D_b': D_b, 'L_b': L_b, 'C': C, 'u_in': u_in,
            'Δt_spinup': Δt_spinup, 'max_spinup': max_spinup, 'Δt': Δt, 'x_nodes': x_nodes.tolist()} #everything the results depend on
save_file, saved, save = progress_file('GRL_idealized', settings)

for sliding_law in sliding_laws:
    for direction in directions:
        if (sliding_law, direction) in saved:
            continue
        md = flowline_mesh(model(), x_nodes)
        md = initialize_model(md, moving_front = False, bed = beds[direction], thickness = thicknesses[direction], **params)

        V = ice_volume(md)
        for step in progress(range(int(round(max_spinup/Δt_spinup))), desc = f'{sliding_law}, {direction} (steps of {Δt_spinup} a)'):
            u, H, s = coupled_solve(md, Δt_spinup, friction = friction(md, sliding_law, C))
            V, V_old = ice_volume(md), V
            if abs(V - V_old)/V/Δt_spinup < 1e-5:
                break
        years = (step + 1)*Δt_spinup

        x_p, ū = profile(md, depth_average(md, u[0]))
        u_t = np.interp(L_0, x_p, ū)
        base_t = np.interp(L_0, *profile(md, md.geometry.base))
        H_t = np.interp(L_0, *profile(md, md.geometry.thickness))
        k_0, f_0 = u_t/(0 - base_t), (base_t - 0 + ϱ*H_t)/H_t

        def drift(calving_law, k, f_star): #how far the terminus moves in the unperturbed run (m)
            md = flowline_mesh(model(), x_nodes)
            md = initialize_model(md, moving_front = False, bed = beds[direction], thickness = H, **params)
            return perturbation_run(md, sliding_law, C, calving_law, 0, k, f_star, Δt)[0] - L_0
        k, drift_k = calibrate(lambda k: drift('rate law', k, None), k_0, 0.05*k_0, f'{sliding_law}, {direction}: k')
        f_star, drift_f = calibrate(lambda f_star: drift('position law', None, f_star), f_0, 0.005, f'{sliding_law}, {direction}: f*')
        saved[(sliding_law, direction)] = {'years': years, 'H': H, 'k': k, 'f_star': f_star, 'k_0': k_0, 'f_0': f_0, 'drift_k': drift_k, 'drift_f': drift_f}
        np.save(save_file, saved)

for sliding_law in sliding_laws:
    for direction in directions:
        c = saved[(sliding_law, direction)]
        steady = f'steady after {c["years"]:.0f} a' if c['years'] < max_spinup else f'NOT steady after {max_spinup} a'
        print(f'{sliding_law}, {direction}: {steady}; k = {c["k_0"]:.3f} -> {c["k"]:.3f} /a (terminus drift {c["drift_k"]:+.2f} m), '
              f'f* = {c["f_0"]:.3f} -> {c["f_star"]:.3f} ({c["drift_f"]:+.2f} m)')

profiles = {} #the spun-up glaciers on both beds, for Figure 1
for sliding_law in sliding_laws:
    for direction in directions:
        md = flowline_mesh(model(), x_nodes)
        md = initialize_model(md, moving_front = False, bed = beds[direction], thickness = saved[(sliding_law, direction)]['H'], **params)
        profiles[(sliding_law, direction)] = flowline_profile(md, diagnostic_solve(md, friction = friction(md, sliding_law, C)))

progress file: GRL_idealized_3c44a2de.npy
power-law, retrograde: steady after 208 a; k = 0.565 -> 0.565 /a (terminus drift -0.00 m), f* = 0.212 -> 0.212 (-0.01 m)
power-law, prograde: steady after 210 a; k = 0.566 -> 0.566 /a (terminus drift -0.00 m), f* = 0.208 -> 0.208 (-0.01 m)
effective-pressure, retrograde: steady after 213 a; k = 0.583 -> 0.583 /a (terminus drift -0.00 m), f* = 0.191 -> 0.191 (-0.01 m)
effective-pressure, prograde: steady after 214 a; k = 0.589 -> 0.589 /a (terminus drift -0.00 m), f* = 0.180 -> 0.180 (-0.01 m)


## Lake-level perturbations

Each run starts from the spun-up glacier. Besides the terminus speed, the speed `speed_distance` behind the initial terminus is recorded, as in plan view (runs saved without it are made again). The printout compares the two beds for the lake 10 m higher; the figures show the retrograde bed.

In [7]:
for sliding_law in sliding_laws:
    for direction in directions:
        for calving_law in calving_laws:
            r = saved.setdefault((sliding_law, direction, calving_law), {'L': [], 'u_t': [], 'flux': []})
            if 'u_100' not in r: #runs saved without the speed behind the terminus are made again
                r.update({'L': [], 'u_t': [], 'flux': [], 'u_100': []})
            for offset in progress(offsets[len(r['L']):], desc = f'{sliding_law}, {direction}, {calving_law}', disable = len(r['L']) == len(offsets)):
                spinup = saved[(sliding_law, direction)]
                md = flowline_mesh(model(), x_nodes)
                md = initialize_model(md, moving_front = False, bed = beds[direction], thickness = spinup['H'], **params)
                L, u_t, flux = perturbation_run(md, sliding_law, C, calving_law, offset, spinup['k'], spinup['f_star'], Δt)
                r['L'].append(L)
                r['u_t'].append(u_t)
                r['flux'].append(flux)
                x_p, ū = profile(md, depth_average(md, md.initialization.vx)) #the final state's velocity
                r['u_100'].append(np.interp(L_0 - speed_distance, x_p, ū)) #speed_distance behind the initial terminus, as in plan view
                save()

def change_at(r, key, offset): #change from the unperturbed run at a lake-level offset
    y = np.array(r[key])
    return (y[offsets == offset] - y[offsets == 0])[0]
print('Lake 10 m higher, retrograde / prograde bed (the figures show the retrograde bed):')
for sliding_law in sliding_laws:
    for calving_law in calving_laws:
        r, p = [saved[(sliding_law, direction, calving_law)] for direction in ['retrograde', 'prograde']]
        print(f'  {sliding_law}, {calving_law}: ' + ', '.join(f'{name} {change_at(r, key, 10):+.{digits}f} / {change_at(p, key, 10):+.{digits}f} {unit}' for key, name, digits, unit in
                                                             [('L', 'Δ terminus position', 1, 'm'), ('u_100', 'Δ speed', 1, 'm/a'), ('flux', 'Δ flux', 0, 'm²/a')]))

Lake 10 m higher, retrograde / prograde bed (the figures show the retrograde bed):
  power-law, rate law: Δ terminus position -7.8 / -7.6 m, Δ speed -3.4 / -3.3 m/a, Δ flux -440 / -409 m²/a
  power-law, position law: Δ terminus position -60.1 / -57.0 m, Δ speed +7.6 / +6.3 m/a, Δ flux +885 / +712 m²/a
  power-law, trivial law: Δ terminus position +0.0 / +0.0 m, Δ speed -4.5 / -4.3 m/a, Δ flux -598 / -555 m²/a
  effective-pressure, rate law: Δ terminus position -4.6 / -4.2 m, Δ speed +1.3 / +1.6 m/a, Δ flux -57 / -10 m²/a
  effective-pressure, position law: Δ terminus position -67.9 / -65.2 m, Δ speed +11.3 / +9.7 m/a, Δ flux +1042 / +833 m²/a
  effective-pressure, trivial law: Δ terminus position +0.0 / +0.0 m, Δ speed +1.0 / +1.4 m/a, Δ flux -103 / -38 m²/a


# Idealized glacier in plan view: buttressing by the valley walls

The retrograde glacier in a valley 500 m wide, in two versions that differ only in how the ice meets the valley walls: frozen to them (no slip), so that the walls hold the glacier back through lateral drag, or sliding freely along them (free slip: the ice cannot flow through the walls, but nothing resists its flow along them). Comparing the two isolates the buttressing that the lateral stresses provide, which a flowline model cannot represent, and how it shapes the response to the lake level. Along the centreline, the valley floor is the flowline bed with the trough behind the terminus; across the valley it rises by `D_v` towards the walls as a parabola, and the walls themselves are vertical. The ice enters at 10 m/a across the head of the valley, and the lake fills the valley beyond the terminus.

The model is the flowline's HO model, but the front spans the valley and moves with ISSM's level set, under the same calving laws applied locally (`issm_helper`'s `rate_law`, at $k$ times the local water depth, normal to the front; `position_law`, removing the ice wherever $f_{af} < f^*$, working back from the front). Each run ends with the mean glacier length (ice area divided by the valley width), the speed `speed_distance` behind the initial front within 100 m of the centreline, and the flux through the front per unit width. The models run on `n_proc` MPI processes (solver, mesh and running times in *Notes*).

## Parameters

In [8]:
W = 500 #valley width (m)
D_v = 40 #rise of the valley floor from the centreline to the walls (m)

x_nodes_2d = np.unique(np.r_[np.arange(0, 4500, 100), np.arange(4500, 4850, 50), np.arange(4850, 5080, 5),
                             np.arange(5080, 5200, 20), np.arange(5200, 5500 + 1, 50)]) #along the valley, with 500 m of lake (m)
y_half = np.array([0, 50, 100, 150, 190, 220, 240, 250])
y_nodes_2d = np.unique(np.r_[-y_half, y_half]) #across the valley (m)
walls_2d = ['no slip', 'free slip'] #how the ice meets the valley walls
max_spinup_2d = 1000 #longest plan-view spin-up (a): the plan-view glaciers settle more slowly than the flowline (with no-slip walls, the volume still changed by 3e-5 per year after 400 a)
offsets_2d = offsets #lake-level perturbations in plan view (m); every other one, offsets[::2], saves a third of the time
paths_2d = [np.c_[np.arange(0, x_nodes_2d[-1] + 1), np.full(int(x_nodes_2d[-1]) + 1, y_0)] for y_0 in np.arange(-100, 101, 10)] #lines along the valley within 100 m of the centreline, every metre: the speed near the terminus is measured on them

## Geometry

`new_model_2d` builds a plan-view model of the glacier with either wall condition from a thickness field, with the front fixed (trivial law) or moving, and an initial guess for the velocity (the depth-averaged velocity of the reference state, which saves the stress balance many iterations). On the walls, both velocity components are zero for no slip; for free slip, only the component across the valley is, and the flow along the walls is free.

In [9]:
md = valley_mesh(model(), x_nodes_2d, y_nodes_2d)
x2, y2 = md.mesh.x, md.mesh.y
s_t2 = np.maximum(L_0 - x2, 0) #distance upstream of the terminus (m)
plane2 = -d_0 + slope*s_t2
trough2 = np.where(s_t2 < L_b, D_b*(1 - np.cos(2*np.pi*s_t2/L_b))/2, 0)
valley2 = D_v*(2*y2/W)**2
bed_2d = plane2 - trough2 + valley2 #the retrograde bed, rising towards the walls

on_walls = np.isclose(np.abs(y2), W/2)
points_2d = points_behind(x2, y2, md.mesh.elements - 1, x2 - L_0, paths_2d, speed_distance) #where the speed near the terminus is measured
params_2d = {'approximation': 'HO',
             'vertical_layers': vertical_layers,
             'temperature': T,
             'friction': C,
             'm': m,
             'rho_water': ρ_w,
             'ice_levelset': x2 - L_0}
spcv_2d = {'no slip': (dirichlet(md, (x2 == 0, u_in), (on_walls, 0)), dirichlet(md, (on_walls | (x2 == 0), 0))),
           'free slip': (dirichlet(md, (x2 == 0, u_in)), dirichlet(md, (on_walls | (x2 == 0), 0)))} #(spcvx, spcvy) for each wall condition

def new_model_2d(wall, H, calving_law, velocity = (0, 0)):
    md = valley_mesh(model(), x_nodes_2d, y_nodes_2d)
    md.cluster.np = n_proc
    spcvx, spcvy = spcv_2d[wall]
    return initialize_model(md, bed = bed_2d, thickness = H, moving_front = calving_law != 'trivial law', velocity = velocity,
                            spcvx = spcvx, spcvy = spcvy, spcthickness = dirichlet(md, (x2 == 0, H)), **params_2d)

## Spin-up and calibration

Both versions of each glacier start from the same geometry, the spun-up flowline glacier's surface (retrograde bed) flat across the valley, and each is spun up with its own walls and the front fixed until the ice volume changes by less than $10^{-5}$ per year, for at most `max_spinup_2d` years. The spin-up is saved every 10 model years and each calibration try as it is made, so a restart carries on from there. $k$ and $f^*$ are then calibrated for zero drift of the mean length over the unperturbed year, starting from $k$ = mean speed at the front / mean water depth there and $f^*$ = the lowest $f_{af}$ along the front (where the position law would calve first).

In [10]:
settings_2d = {'flowline': settings, 'bed': 'retrograde', 'walls': walls_2d, 'max_spinup_2d': max_spinup_2d, 'W': W, 'D_v': D_v, 'x_nodes_2d': x_nodes_2d.tolist(),
               'y_nodes_2d': y_nodes_2d.tolist(), 'offsets_2d': np.asarray(offsets_2d).tolist()} #everything the results depend on
save_file_2d, saved_2d, save_2d = progress_file('GRL_idealized_2d', settings_2d)

md_fl = flowline_mesh(model(), x_nodes)
n_spinup = int(round(max_spinup_2d/Δt_spinup))
for sliding_law in sliding_laws:
    for wall in walls_2d:
        if (sliding_law, wall) in saved_2d:
            continue
        #the spin-up so far, or else the spun-up flowline glacier (grounded everywhere) with its surface flat across the valley
        state = saved_2d.get(('spin-up', sliding_law, wall))
        if state is None:
            x_s, surface = profile(md_fl, beds['retrograde'] + saved[(sliding_law, 'retrograde')]['H'])
            state = {'H': np.maximum(np.interp(x2, x_s, surface) - bed_2d, 1), 'velocity': (0, 0), 'steps': 0, 'done': False}
        md = new_model_2d(wall, state['H'], 'trivial law', state['velocity'])
        V = ice_volume(md)
        for step in ([] if state['done'] else progress(range(state['steps'], n_spinup), initial = state['steps'], total = n_spinup,
                                                            desc = f'{sliding_law}, {wall} (steps of {Δt_spinup} a)')):
            u, H, s = coupled_solve(md, Δt_spinup, friction = friction(md, sliding_law, C))
            V, V_old = ice_volume(md), V
            volume_change = abs(V - V_old)/V/Δt_spinup #relative change per year
            state = {'H': H, 'velocity': (depth_average(md, u[0]), depth_average(md, u[1])), 'steps': step + 1, 'volume_change': volume_change,
                     'done': volume_change < 1e-5 or step + 1 == n_spinup}
            if state['done'] or (step + 1) % 20 == 0: #every 10 years, so that a restart carries on from here
                saved_2d[('spin-up', sliding_law, wall)] = state
                save_2d()
            if state['done']:
                break
        years = state['steps']*Δt_spinup

        #the reference state's front, the starting guesses and every calibration try, kept so that a restart replays them
        reference = saved_2d.setdefault(('reference', sliding_law, wall), {})
        if 'k_0' not in reference:
            u = diagnostic_solve(md, friction = friction(md, sliding_law, C))
            f = front_diagnostics(md)
            n2 = md.mesh.numberofvertices2d
            on_front = np.isclose(md.mesh.x[:n2], L_0)
            order = np.argsort(md.mesh.y[:n2][on_front])
            y_f, b_f, H_f = md.mesh.y[:n2][on_front][order], md.geometry.base[:n2][on_front][order], md.geometry.thickness[:n2][on_front][order]
            depth = np.sum((-b_f[1:] - b_f[:-1])/2*np.diff(y_f))/W #mean water depth along the front
            reference.update({'velocity': (depth_average(md, u[0]), depth_average(md, u[1])), 'length': f.area/W,
                              'k_0': f.speed/depth, 'f_0': np.min((b_f - 0 + ϱ*H_f)/H_f), #f_0 where the position law would calve first
                              'k tries': {}, 'f* tries': {}})
            save_2d()
        H, velocity = state['H'], reference['velocity']

        def drift(calving_law, k, f_star): #how far the mean length changes in the unperturbed run (m)
            run = perturbation_run_2d(lambda law: new_model_2d(wall, H, law, velocity), sliding_law, C, calving_law, 0, k, f_star, Δt, W, points_2d)
            return run[0] - reference['length']
        k, drift_k = calibrate(remembered(lambda k: drift('rate law', k, None), reference['k tries'], save_2d),
                               reference['k_0'], 0.05*reference['k_0'], f'{sliding_law}, {wall}: k')
        f_star, drift_f = calibrate(remembered(lambda f_star: drift('position law', None, f_star), reference['f* tries'], save_2d),
                                    reference['f_0'], 0.005, f'{sliding_law}, {wall}: f*')
        saved_2d[(sliding_law, wall)] = {'years': years, 'volume_change': state['volume_change'], 'H': H, 'velocity': velocity,
                                              'length': reference['length'], 'k': k, 'f_star': f_star, 'k_0': reference['k_0'], 'f_0': reference['f_0'],
                                              'drift_k': drift_k, 'drift_f': drift_f}
        save_2d()

for sliding_law in sliding_laws:
    for wall in walls_2d:
        c = saved_2d[(sliding_law, wall)]
        steady = f'steady after {c["years"]:.0f} a' if c['years'] < max_spinup_2d else f'NOT steady after {max_spinup_2d} a (volume changing by {c["volume_change"]:.1e} per year)'
        print(f'{sliding_law}, {wall}: {steady}; k = {c["k_0"]:.3f} -> {c["k"]:.3f} /a (drift {c["drift_k"]:+.2f} m), '
              f'f* = {c["f_0"]:.3f} -> {c["f_star"]:.3f} ({c["drift_f"]:+.2f} m)')

progress file: GRL_idealized_2d_d3ff7f50.npy
power-law, no slip: steady after 482 a; k = 0.611 -> 0.580 /a (drift -0.21 m), f* = 0.226 -> 0.272 (+0.83 m)
power-law, free slip: steady after 227 a; k = 0.773 -> 0.735 /a (drift +0.26 m), f* = 0.234 -> 0.294 (+0.03 m)
effective-pressure, no slip: steady after 486 a; k = 0.627 -> 0.596 /a (drift +0.03 m), f* = 0.208 -> 0.256 (+0.29 m)
effective-pressure, free slip: steady after 230 a; k = 0.790 -> 0.751 /a (drift +0.25 m), f* = 0.219 -> 0.281 (-0.12 m)


## Lake-level perturbations in plan view

Each run starts from the spun-up glacier; progress is saved after every run. Runs saved with another measure of the speed are made again (the spin-ups and calibrations are kept).

In [11]:
for sliding_law in sliding_laws:
    for wall in walls_2d:
        spinup = saved_2d[(sliding_law, wall)]
        for calving_law in calving_laws:
            r = saved_2d.setdefault((sliding_law, wall, calving_law), {'L': [], 'u_t': [], 'flux': []})
            if r.get('speed') != f'{speed_distance} m behind the initial front': #runs made with another speed measure are made again
                r.update({'L': [], 'u_t': [], 'flux': [], 'speed': f'{speed_distance} m behind the initial front'})
            for offset in progress(offsets_2d[len(r['L']):], desc = f'{sliding_law}, {wall}, {calving_law}', disable = len(r['L']) == len(offsets_2d)):
                L, u_t, flux = perturbation_run_2d(lambda law: new_model_2d(wall, spinup['H'], law, spinup['velocity']), sliding_law, C, calving_law,
                                                   offset, spinup['k'], spinup['f_star'], Δt, W, points_2d)
                r['L'].append(L)
                r['u_t'].append(u_t)
                r['flux'].append(flux)
                save_2d()

# Luggye Glacier (flowline)

## Data and parameters

The RGI 7.0 centreline of Luggye, and the drone DEM. The flowline covers the lower part of the glacier, in the ablation zone; SMB is neglected in the model, and ice enters at its top at a prescribed speed; the bed inversion assumes the glacier is thinning (see *Bed inversion*). $C$ is icepack's 0.1 MPa (m/a)$^{-1/3}$; friction and the inflow speed are the same in the inversion and the perturbation runs.

In [12]:
data_dir = '/gpfs/home3/cbezu/issm/notebooks/data'
centerline_path = data_dir + '/centerlines/RGI2000-v7.0-L-15_south_asia_east/RGI2000-v7.0-L-15_south_asia_east.shp'
dem_path = data_dir + '/dems/Luggye_2025_ds.tif'
glacier_id = 'RGI2000-v7.0-G-15-09534'

lake_level = 4480 #(m)
C_L = np.sqrt(0.1e6*yts**(1/m)) #Weertman C
u_in_L = 10 #inflow speed at the top of the flowline (m/a)
thinning_L = 4.67 #thinning the bed inversion aims for (m/a): the mean observed on Luggye's ablation area in 2004-2011 (Tsutaki et al., 2019; see Bed inversion)
K = 0.9 #bed correction per metre of surface misfit
Δt_inversion = 0.5 #time step of the inversion's model years (a)
inversion_tolerance = 0.25 #the inversion stops once the RMS misfit is below this (m)
max_iterations = 300 #at most, for each sliding law
Δt_L = 1/12 #perturbation time step (a)

## Flowline

The centreline, from the top of the DEM to where the surface comes within 20 m of the lake, with the surface sampled on 50 cells, smoothed over three cells and taken relative to the lake.

In [13]:
metric_crs = 'EPSG:32645' #distances along the flowline are measured in UTM 45N, whatever the DEM's projection
centerlines = gpd.read_file(centerline_path)
line = centerlines[centerlines.rgi_g_id == glacier_id].to_crs(metric_crs).geometry.values[0]

with rasterio.open(dem_path) as dem:
    to_dem = Transformer.from_crs(metric_crs, dem.crs, always_xy = True)
    d = np.arange(0, line.length, 5) #distance along the centreline (m)
    x_d, y_d = to_dem.transform(*np.array([line.interpolate(v).coords[0] for v in d]).T)
    z = np.array([v.astype(float).filled(np.nan)[0] for v in dem.sample(zip(x_d, y_d), masked = True)]) #missing data as NaN
    if np.all(np.isnan(z)):
        raise ValueError(f'the DEM ({dem.crs}, bounds {tuple(round(b) for b in dem.bounds)}) has no data along the centreline')
    top = d[~np.isnan(z)][0]
    end = d[~np.isnan(z) & (z < lake_level + 20)][0]
    line = substring(line, top, end)
    x_L = np.linspace(0, line.length, 51)
    x_d, y_d = to_dem.transform(*np.array([line.interpolate(v).coords[0] for v in x_L]).T)
    s_obs = np.array([v.astype(float).filled(np.nan)[0] for v in dem.sample(zip(x_d, y_d), masked = True)])

s_obs = scipy.ndimage.gaussian_filter1d(s_obs, 1.5) - lake_level
L_L = x_L[-1] #glacier length (m)
print(f'flowline {L_L:.0f} m long; surface {s_obs[0]:.0f} m above the lake at the top, {s_obs[-1]:.0f} m at the terminus')

flowline 2120 m long; surface 801 m above the lake at the top, 34 m at the terminus


## Bed inversion

After van Pelt et al. (2013), for each sliding law: starting from 200 m of ice, model one year with the front fixed, lower the bed where the surface ends above its target and raise it where it ends below ($\Delta b = -K\,(s - s_{target})$), and repeat until the RMS misfit is below `inversion_tolerance`. The friction is evaluated at the start of each model year, which has two steps of `Δt_inversion`.

The target is the observed surface lowered by a year's thinning, `thinning_L` = 4.67 m/a, since Luggye is far from balance: in 2004–2011, its debris-covered ablation area thinned by 4.67 m/a on average (Tsutaki et al., 2019, who call it Lugge Glacier). With no surface mass balance in the model, a glacier in balance could discharge at its front only what enters at its top, and the terminus would end up in shallow water; thinning at the observed rate, it ends up in 60–80 m of water, depending on the sliding law (the lake is 50 m deep on average and 126 m at its deepest; Tsutaki et al., 2019). As in the icepack version, the thickness at the top of the flowline is free during the inversion, so the bed there is corrected like everywhere else; this sets how much ice enters. The plan view uses this bed (see *The bed* there).

In [14]:
settings_L = {'calibration': 'zero drift', 'flux': 'at the final position', 'vertical_layers': vertical_layers, 'T': T, 'A': float(issm_helper.rate_factor(T)), 'm': m, 'ρ_w': ρ_w, 'offsets': offsets.tolist(), 'ramp_time': ramp_time,
              'centerline_path': centerline_path, 'dem_path': dem_path, 'glacier_id': glacier_id,
              'lake_level': lake_level, 'C_L': C_L, 'u_in_L': u_in_L, 'K': K, 'Δt_L': Δt_L,
              'thinning_L': thinning_L, 'Δt_inversion': Δt_inversion, 'inversion_tolerance': inversion_tolerance, 'max_iterations': max_iterations} #everything the results depend on
save_file_L, saved_L, save_L = progress_file('GRL_luggye', settings_L)

md = flowline_mesh(model(), x_L)
x = md.mesh.x
s_nodes = np.interp(x, x_L, s_obs)
params_inversion = {'approximation': 'HO',
                    'vertical_layers': vertical_layers,
                    'temperature': T,
                    'friction': C_L,
                    'm': m,
                    'rho_water': ρ_w,
                    'moving_front': False,
                    'ice_levelset': x - L_L,
                    'spcvx': dirichlet(md, (x == 0, u_in_L))}

for sliding_law in sliding_laws:
    if ('bed', sliding_law) in saved_L:
        continue
    bed = s_nodes - 200
    for iteration in progress(range(max_iterations), desc = sliding_law):
        md = flowline_mesh(model(), x_L)
        md = initialize_model(md, bed = bed, thickness = s_nodes - bed, **params_inversion)
        u, H, s = coupled_solve(md, Δt_inversion, num_steps = int(round(1/Δt_inversion)), friction = friction(md, sliding_law, C_L))
        misfit = s - (s_nodes - thinning_L) #against the observed surface lowered by a year's thinning
        bed = bed - K*misfit
        if np.sqrt(np.mean(misfit**2)) < inversion_tolerance:
            break
    saved_L[('bed', sliding_law)] = profile(md, bed)[1] #along x_L
    saved_L[('inversion', sliding_law)] = {'iterations': iteration + 1, 'misfit': np.sqrt(np.mean(misfit**2)), 'thinning': np.mean(s_nodes - s)}
    np.save(save_file_L, saved_L)
for sliding_law in sliding_laws:
    b = saved_L[('inversion', sliding_law)]
    print(f'{sliding_law}: bed after {b["iterations"]} iterations (RMS misfit {b["misfit"]:.2f} m); in its last model year, the glacier thinned by {b["thinning"]:.2f} m/a '
          f'on average (target {thinning_L} m/a)')

progress file: GRL_luggye_37343bb2.npy
power-law: bed after 15 iterations (RMS misfit 0.24 m); in its last model year, the glacier thinned by 4.68 m/a on average (target 4.67 m/a)
effective-pressure: bed after 18 iterations (RMS misfit 0.25 m); in its last model year, the glacier thinned by 4.61 m/a on average (target 4.67 m/a)


## Reference state and calibration

The observed surface on the inverted bed, on a mesh extended 2 km into the lake, with $k$ and $f^*$ calibrated for zero terminus drift. Nodes are 1 m apart within 50 m of the terminus, 5 m apart from 400 m behind it to 100 m beyond it, and 25 m apart elsewhere. The lake floor beyond the terminus stays at the terminus depth. The top of the flowline keeps its thickness and inflow.

In [15]:
x_nodes_L = np.r_[np.linspace(0, L_L - 400, 70), L_L + np.r_[np.arange(-395, -50, 5), np.arange(-50, 50), np.arange(50, 100, 5), np.arange(100, 2000 + 1, 25)]] #node positions (m)
md = flowline_mesh(model(), x_nodes_L)
x = md.mesh.x
params_L = {'approximation': 'HO',
            'vertical_layers': vertical_layers,
            'temperature': T,
            'friction': C_L,
            'm': m,
            'rho_water': ρ_w,
            'moving_front': False,
            'ice_levelset': x - L_L,
            'spcvx': dirichlet(md, (x == 0, u_in_L))}

references = {}
for sliding_law in sliding_laws:
    bed = np.interp(x, x_L, saved_L[('bed', sliding_law)])
    H = np.interp(x, x_L, s_obs) - bed
    md = flowline_mesh(model(), x_nodes_L)
    md = initialize_model(md, bed = bed, thickness = H, spcthickness = dirichlet(md, (x == 0, H)), **params_L)
    u = diagnostic_solve(md, friction = friction(md, sliding_law, C_L))

    x_p, ū = profile(md, depth_average(md, u[0]))
    u_t = np.interp(L_L, x_p, ū)
    base_t = np.interp(L_L, *profile(md, md.geometry.base))
    H_t = np.interp(L_L, *profile(md, md.geometry.thickness))
    if ('calibration', sliding_law) not in saved_L:
        k_0, f_0 = u_t/(0 - base_t), (base_t - 0 + ϱ*H_t)/H_t

        def drift(calving_law, k, f_star): #how far the terminus moves in the unperturbed run (m)
            md = flowline_mesh(model(), x_nodes_L)
            md = initialize_model(md, bed = bed, thickness = H, spcthickness = dirichlet(md, (x == 0, H)), **params_L)
            return perturbation_run(md, sliding_law, C_L, calving_law, 0, k, f_star, Δt_L)[0] - L_L
        k, drift_k = calibrate(lambda k: drift('rate law', k, None), k_0, 0.05*k_0, f'{sliding_law}: k')
        f_star, drift_f = calibrate(lambda f_star: drift('position law', None, f_star), f_0, 0.005, f'{sliding_law}: f*')
        saved_L[('calibration', sliding_law)] = {'k': k, 'f_star': f_star, 'k_0': k_0, 'f_0': f_0, 'drift_k': drift_k, 'drift_f': drift_f}
        np.save(save_file_L, saved_L)

    c = saved_L[('calibration', sliding_law)]
    references[sliding_law] = {'bed': bed, 'H': H, 'md': md, 'u': u, 'k': c['k'], 'f_star': c['f_star'], 'profile': flowline_profile(md, u)}
    print(f'{sliding_law}: terminus in {0 - base_t:.0f} m of water, {H_t:.0f} m thick, at {u_t:.1f} m/a; '
          f'k = {c["k_0"]:.3f} -> {c["k"]:.3f} /a (terminus drift {c["drift_k"]:+.2f} m), f* = {c["f_0"]:.3f} -> {c["f_star"]:.3f} ({c["drift_f"]:+.2f} m)')

power-law: terminus in 76 m of water, 109 m thick, at 109.2 m/a; k = 1.445 -> 1.260 /a (terminus drift -0.10 m), f* = 0.226 -> 0.183 (-0.45 m)


effective-pressure: terminus in 58 m of water, 92 m thick, at 126.7 m/a; k = 2.170 -> 1.955 /a (terminus drift +0.17 m), f* = 0.283 -> 0.243 (+0.08 m)


## Lake-level perturbations

Besides the terminus speed, the speed `speed_distance` behind the initial terminus is recorded, as in plan view (runs saved without it are made again).

In [16]:
for sliding_law in sliding_laws:
    reference = references[sliding_law]
    for calving_law in calving_laws:
        r = saved_L.setdefault((sliding_law, calving_law), {'L': [], 'u_t': [], 'flux': []})
        if 'u_100' not in r: #runs saved without the speed behind the terminus are made again
            r.update({'L': [], 'u_t': [], 'flux': [], 'u_100': []})
        for offset in progress(offsets[len(r['L']):], desc = f'{sliding_law}, {calving_law}', disable = len(r['L']) == len(offsets)):
            md = flowline_mesh(model(), x_nodes_L)
            md = initialize_model(md, bed = reference['bed'], thickness = reference['H'], spcthickness = dirichlet(md, (x == 0, reference['H'])), **params_L)
            L, u_t, flux = perturbation_run(md, sliding_law, C_L, calving_law, offset, reference['k'], reference['f_star'], Δt_L)
            r['L'].append(L)
            r['u_t'].append(u_t)
            r['flux'].append(flux)
            x_p, ū = profile(md, depth_average(md, md.initialization.vx)) #the final state's velocity
            r['u_100'].append(np.interp(L_L - speed_distance, x_p, ū)) #speed_distance behind the initial terminus, as in plan view
            np.save(save_file_L, saved_L)

# Luggye Glacier in plan view

Luggye in plan view, with the same choices as the flowline: the drone DEM, the lake level, $C$, the inflow speed and the flowline's inverted bed, now on the glacier's outline (the RGI 7.0 outline cut by the DEM, extended into the lake, clipped to the DEM's coverage). The terminus is the front mapped from imagery of 20 October 2025, two days after the DEM. Ice enters across the top of the DEM where the consensus thickness (Farinotti et al., 2019) counts it as glacier, at 10 m/a in the middle (the flowline's inflow speed), and is frozen to the valley sides everywhere else (see *Notes*). The front moves as in the idealized plan view.

## Data and parameters

In [17]:
outline_file = data_dir + '/exp_files/DomainOutline_20260126_162319.exp' #glacier outline, extended into the lake (UTM 45N)
thickness_file = data_dir + '/bed/thickness_RGI60-15.02373.tif' #consensus ice thickness (Farinotti et al., 2019): where ice enters
terminus_lonlat = [(90.31877330330583, 28.09807611129586), (90.31919172791216, 28.098104504384743), (90.31958869484636, 28.0979909319842),
                   (90.31985691574785, 28.09783950192981), (90.32003930596086, 28.097602892042097), (90.32007149246904, 28.097356817205473),
                   (90.3201358654854, 28.097176992929565), (90.32029679802629, 28.096940381581167), (90.32055429009172, 28.09669430522543),
                   (90.32076886681291, 28.096438763797273), (90.32100490120622, 28.096154828163456), (90.32110146073076, 28.095889820894154),
                   (90.32110146073076, 28.095539631713223), (90.32101563004228, 28.095113724412702)] #terminus on 20 October 2025 (GEEDiT)

h_fine, h_coarse = 10, 50 #element sizes (m): at the terminus and elsewhere
r_fine, r_grade = 80, 250 #elements are finest within r_fine of the terminus and coarsen over r_grade (m)
smoothing = 3 #width of the Gaussian smoothing of the DEM (pixels), as the flowline smooths its profile
offsets_L2 = offsets #lake-level perturbations in plan view (m)

## Domain and meshes

`luggye_mesh` meshes the domain with `bamg`, refined around the terminus; `luggye_fields` puts the data on a mesh: the observed surface relative to the lake, the consensus thickness, the ice level set (the signed distance to the terminus, negative upstream, zero on the vertices of the front; a front vertex that belongs to no element with ice is ice-free, as ISSM treats it), the inflow and the valley sides, and the inward direction of the inflow.

In [18]:
with contextlib.redirect_stdout(io.StringIO()): #ISSM's note that it has no shapefile reader (not needed here)
    from bamg import bamg
from shapely.geometry import Polygon, LineString, Point, shape
from shapely.ops import split
from shapely import points, distance as shapely_distance, contains_xy
from rasterio.features import shapes
import rasterio.warp
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

lines = [l for l in open(outline_file).read().splitlines() if not l.startswith('#')]
outline = Polygon(np.array([[float(v) for v in l.split()] for l in lines[1:] if l.strip()]))
with rasterio.open(dem_path) as dem:
    z_dem = dem.read(1, masked = True).astype(float).filled(np.nan)
    valid = np.isfinite(z_dem)
    coverage = max((shape(g) for g, v in shapes(valid.astype(np.uint8), mask = valid, transform = dem.transform) if v == 1), key = lambda g: g.area)
    pixel, dem_transform, dem_crs = dem.res[0], dem.transform, dem.crs
domain = outline.intersection(coverage.buffer(-pixel/2))
domain = (max(domain.geoms, key = lambda g: g.area) if domain.geom_type == 'MultiPolygon' else domain).simplify(5)

#the DEM smoothed, with gaps filled from the nearest data first (the domain lies inside the data, so this only matters at its edge)
filled = z_dem[tuple(scipy.ndimage.distance_transform_edt(~valid, return_distances = False, return_indices = True))]
z_smooth = scipy.ndimage.gaussian_filter(filled, smoothing)

terminus = LineString(gpd.GeoSeries([LineString(terminus_lonlat)], crs = 'EPSG:4326').to_crs(metric_crs).iloc[0].coords)
c_t = np.array(terminus.coords)
ends = [c_t[0] + (c_t[0] - c_t[1])/np.linalg.norm(c_t[0] - c_t[1])*500, c_t[-1] + (c_t[-1] - c_t[-2])/np.linalg.norm(c_t[-1] - c_t[-2])*500]
cutter = LineString(np.vstack([ends[0], c_t, ends[1]])) #the terminus, extended across the domain
ice_part = sorted(split(domain, cutter).geoms, key = lambda g: -g.centroid.y)[0] #upstream of the terminus (the glacier flows south)
W_L = terminus.length #length of the observed terminus (m), for lengths and fluxes per unit width
def continued(l): #a line, continued 500 m into the lake, as points every metre
    c = np.array(l.coords)
    l = LineString(np.vstack([c, c[-1] + (c[-1] - c[-2])/np.linalg.norm(c[-1] - c[-2])*500]))
    return np.array([l.interpolate(v).coords[0] for v in np.arange(0, l.length, 1.0)])
paths_L2 = [continued(line.offset_curve(d)) for d in np.arange(-100, 101, 10)] #the flowline and lines within 100 m of it: the speed near the terminus is measured on them

def luggye_mesh(h_terminus):
    outer = np.array(domain.exterior.coords)
    with contextlib.redirect_stdout(io.StringIO()): #bamg's notes (a contour re-oriented)
        md = bamg(model(), 'domain', [{'x': outer[:, 0], 'y': outer[:, 1], 'nods': len(outer), 'closed': 1, 'density': 1, 'name': 'domain', 'Geometry': 'Polygon'}],
                  'hmax', h_coarse)
        for refinement in range(2):
            d = shapely_distance(points(md.mesh.x, md.mesh.y), terminus)
            h = np.clip(h_terminus + (h_coarse - h_terminus)*np.clip((d - r_fine)/r_grade, 0, 1), h_terminus, h_coarse)
            md = bamg(md, 'hVertices', h, 'hmax', h_coarse, 'hmin', h_terminus)
    return md

def luggye_fields(md):
    x, y = md.mesh.x, md.mesh.y
    elements = md.mesh.elements.astype(int) - 1
    cols, rows = ~dem_transform*(x, y) #fractional pixel positions
    s = scipy.ndimage.map_coordinates(z_smooth, [rows - 0.5, cols - 0.5], order = 1, mode = 'nearest') - lake_level #bilinear, between pixel centres
    with rasterio.open(thickness_file) as r:
        xs, ys = rasterio.warp.transform(metric_crs, r.crs, x, y)
        H_consensus = np.nan_to_num(np.array([v[0] for v in r.sample(zip(xs, ys), masked = True)], dtype = float))
    ice = contains_xy(ice_part.buffer(1e-6), x, y)
    φ = shapely_distance(points(x, y), cutter)*np.where(ice, -1., 1.)
    cut = np.any(ice[elements], axis = 1) & np.any(~ice[elements], axis = 1)
    front = np.zeros(x.size, dtype = bool)
    front[elements[cut].ravel()] = True
    front &= ice
    φ[front] = 0. #the front on the outermost ice vertices
    ice_elements = np.min(φ[elements], axis = 1) < 0 #as ISSM counts them
    alone = ice & ~np.isin(np.arange(x.size), elements[ice_elements]) #front vertices in no element with ice: ice-free
    φ[alone], ice[alone], front[alone] = np.abs(φ[alone]) + 1e-3, False, False
    segments = md.mesh.segments.astype(int)[:, :2] - 1
    boundary = np.zeros(x.size, dtype = bool)
    boundary[segments.ravel()] = True
    inflow = boundary & ice & (shapely_distance(points(x, y), coverage.exterior) < 2*pixel) & (s > 200) & (H_consensus > 0)
    #the inflow: the longest stretch of boundary on the edge of the DEM (its top) that the consensus counts as glacier, at u_in_L in the
    #middle, falling to zero at its ends as a parabola; the rest of the top edge (a flat bench outside the consensus glacier, where ice
    #pushed in piles up) and shorter stretches (the tributary's narrow notch, where it has nowhere to go) are valley sides
    stretches = segments[inflow[segments[:, 0]] & inflow[segments[:, 1]]]
    label = connected_components(coo_matrix((np.ones(len(stretches)), tuple(stretches.T)), shape = (x.size, x.size)), directed = False)[1]
    paths = []
    for c in np.unique(label[inflow]):
        members = np.flatnonzero(inflow & (label == c))
        neighbours = {v: [b if a == v else a for a, b in stretches if v in (a, b)] for v in members}
        order = [next((v for v in members if len(neighbours[v]) == 1), members[0])] #from one end along the stretch
        while next_vertex := [v for v in neighbours[order[-1]] if v not in order]:
            order.append(next_vertex[0])
        paths.append((np.r_[0, np.cumsum(np.hypot(np.diff(x[order]), np.diff(y[order])))], order))
    arc, order = max(paths, key = lambda p: p[0][-1])
    inflow_speed = np.zeros(x.size)
    inflow_speed[order] = u_in_L*(1 - (2*arc/arc[-1] - 1)**2)
    inflow &= inflow_speed > 0 #the ends of the stretch are frozen to the sides
    sides = boundary & ~inflow
    normal = np.zeros((x.size, 2)) #inward normal on the boundary
    for a, b in segments:
        n = np.array([y[a] - y[b], x[b] - x[a]])/np.hypot(x[b] - x[a], y[b] - y[a])
        if not domain.buffer(1e-3).contains(Point((x[a] + x[b])/2 + 1e-2*n[0], (y[a] + y[b])/2 + 1e-2*n[1])):
            n = -n
        normal[a] += n
        normal[b] += n
    normal /= np.maximum(np.linalg.norm(normal, axis = 1), 1e-12)[:, None]
    return SimpleNamespace(x = x, y = y, s = s, H_consensus = H_consensus, ice_levelset = φ, ice = ice, front = front,
                           inflow = inflow, sides = sides, spcvx = dirichlet(md, (inflow, inflow_speed*normal[:, 0]), (sides, 0)),
                           spcvy = dirichlet(md, (inflow, inflow_speed*normal[:, 1]), (sides, 0)))

md_L2 = luggye_mesh(h_fine)
fields_L2 = luggye_fields(md_L2)
points_L2 = points_behind(fields_L2.x, fields_L2.y, md_L2.mesh.elements - 1, fields_L2.ice_levelset, paths_L2, speed_distance) #where the speed near the terminus is measured
print(f'mesh: {md_L2.mesh.numberofvertices} vertices, {fields_L2.front.sum()} on the front; terminus {W_L:.0f} m long')

Construction of a mesh from a given geometry
Anisotropic mesh adaptation

   new number of triangles = 5046
Anisotropic mesh adaptation

   new number of triangles = 5127


mesh: 2669 vertices, 74 on the front; terminus 454 m long


## The bed

The plan view uses the flowline's inverted bed for each sliding law: the ice is as thick as the flowline's at the same distance along the flowline, all the way across the glacier (at least 10 m). The two models then share the bed along the flowline and the water depth at the terminus, and differ only in the second dimension. On this bed the plan-view glacier is not in balance with the observed surface either; the experiments measure changes from its unperturbed year. (A bed inverted in plan view gives no usable glacier: see *Notes*.)

In [19]:
settings_L2 = {'flowline': settings_L, 'outline_file': outline_file, 'thickness_file': thickness_file, 'terminus': terminus_lonlat,
               'h': (h_fine, h_coarse, r_fine, r_grade), 'smoothing': smoothing, 'bed': 'flowline',
               'offsets_L2': np.asarray(offsets_L2).tolist()} #everything the results depend on
save_file_L2, saved_L2, save_L2 = progress_file('GRL_luggye_2d', settings_L2)

params_L2 = {'approximation': 'HO', 'vertical_layers': vertical_layers, 'temperature': T, 'friction': C_L, 'm': m, 'rho_water': ρ_w}

def new_model_L2(md_mesh, fields, bed, H, calving_law, velocity = (0, 0)):
    md = copy.deepcopy(md_mesh)
    md.cluster.np = n_proc
    return initialize_model(md, bed = bed, thickness = H, ice_levelset = fields.ice_levelset, moving_front = calving_law != 'trivial law',
                            velocity = velocity, spcvx = fields.spcvx, spcvy = fields.spcvy, spcthickness = dirichlet(md, (fields.inflow, H)), **params_L2)

def flowline_bed(fields, sliding_law): #the flowline's inverted bed: its thickness at the same distance along the flowline, across the glacier
    along = np.array([line.project(Point(px, py)) for px, py in zip(fields.x, fields.y)])
    return fields.s - np.maximum(np.interp(along, x_L, s_obs - saved_L[('bed', sliding_law)]), 10)

progress file: GRL_luggye_2d_e3515d9a.npy


## Reference state and calibration

The observed surface on the flowline's bed, with the lake floor beyond the terminus at the depth of the nearest point of the front. $k$ and $f^*$ are calibrated for zero change in ice area over the unperturbed year, starting from $k$ = mean speed at the front / mean water depth there and $f^*$ = the lowest $f_{af}$ along the front (where the position law would calve first). Lengths and fluxes are per unit length of the observed terminus. Each calibration try is saved as it is made, so a restart replays them.

In [20]:
from scipy.spatial import cKDTree

f = fields_L2
references_L2 = {}
for sliding_law in sliding_laws:
    bed = flowline_bed(f, sliding_law)
    lake = ~f.ice
    bed[lake] = bed[f.front][cKDTree(np.c_[f.x[f.front], f.y[f.front]]).query(np.c_[f.x[lake], f.y[lake]])[1]]
    H = np.where(f.ice, np.maximum(f.s - bed, 1), 0)

    if ('calibration', sliding_law) not in saved_L2:
        #the reference state's front, the starting guesses and every calibration try, kept so that a restart replays them
        reference = saved_L2.setdefault(('reference', sliding_law), {})
        if 'k_0' not in reference:
            md = new_model_L2(md_L2, f, bed, H, 'trivial law')
            u = diagnostic_solve(md, friction = friction(md, sliding_law, C_L))
            fd = front_diagnostics(md)
            f_af = (md.geometry.base[:md.mesh.numberofvertices2d] - 0 + ϱ*H)/np.maximum(H, 1)
            reference.update({'velocity': (depth_average(md, u[0]), depth_average(md, u[1])), 'area': fd.area, 'depth': np.mean((0 - bed)[f.front]),
                              'thickness': np.mean(H[f.front]), 'speed': fd.speed, 'k tries': {}, 'f* tries': {}})
            reference.update({'k_0': fd.speed/reference['depth'], 'f_0': np.min(f_af[f.front])})
            save_L2()
        new_model = lambda law, bed = bed, H = H, velocity = reference['velocity']: new_model_L2(md_L2, f, bed, H, law, velocity)

        def drift(calving_law, k, f_star): #how far the mean terminus position moves in the unperturbed run (m)
            return perturbation_run_2d(new_model, sliding_law, C_L, calving_law, 0, k, f_star, Δt_L, W_L, points_L2)[0] - reference['area']/W_L
        k, drift_k = calibrate(remembered(lambda k: drift('rate law', k, None), reference['k tries'], save_L2),
                               reference['k_0'], 0.05*reference['k_0'], f'{sliding_law}: k')
        f_star, drift_f = calibrate(remembered(lambda f_star: drift('position law', None, f_star), reference['f* tries'], save_L2),
                                    reference['f_0'], 0.005, f'{sliding_law}: f*')
        saved_L2[('calibration', sliding_law)] = {'k': k, 'f_star': f_star, 'k_0': reference['k_0'], 'f_0': reference['f_0'], 'drift_k': drift_k,
                                                  'drift_f': drift_f, 'velocity': reference['velocity'], 'depth': reference['depth'],
                                                  'thickness': reference['thickness'], 'speed': reference['speed']}
        save_L2()

    c = saved_L2[('calibration', sliding_law)]
    new_model = lambda law, bed = bed, H = H, velocity = c['velocity']: new_model_L2(md_L2, f, bed, H, law, velocity)
    references_L2[sliding_law] = {'bed': bed, 'H': H, 'model': new_model, 'k': c['k'], 'f_star': c['f_star']}
    print(f'{sliding_law}: front in {c["depth"]:.0f} m of water, {c["thickness"]:.0f} m thick, at {c["speed"]:.1f} m/a (means along it); '
          f'k = {c["k_0"]:.3f} -> {c["k"]:.3f} /a (drift {c["drift_k"]:+.2f} m), f* = {c["f_0"]:.3f} -> {c["f_star"]:.3f} ({c["drift_f"]:+.2f} m)')

power-law: front in 77 m of water, 112 m thick, at 67.0 m/a (means along it); k = 0.871 -> 0.836 /a (drift -0.02 m), f* = 0.159 -> 0.202 (-0.33 m)
effective-pressure: front in 59 m of water, 94 m thick, at 80.3 m/a (means along it); k = 1.367 -> 1.299 /a (drift +0.18 m), f* = 0.204 -> 0.250 (-0.12 m)


## Lake-level perturbations in plan view

In [21]:
for sliding_law in sliding_laws:
    reference = references_L2[sliding_law]
    for calving_law in calving_laws:
        r = saved_L2.setdefault((sliding_law, calving_law), {'L': [], 'u_t': [], 'flux': []})
        if r.get('speed') != f'{speed_distance} m behind the initial front': #runs made with another speed measure are made again
            r.update({'L': [], 'u_t': [], 'flux': [], 'speed': f'{speed_distance} m behind the initial front'})
        for offset in progress(offsets_L2[len(r['L']):], desc = f'{sliding_law}, {calving_law}', disable = len(r['L']) == len(offsets_L2)):
            L, u_t, flux = perturbation_run_2d(reference['model'], sliding_law, C_L, calving_law, offset,
                                               reference['k'], reference['f_star'], Δt_L, W_L, points_L2)
            r['L'].append(L)
            r['u_t'].append(u_t)
            r['flux'].append(flux)
            save_L2()

# Figures

All four figures are drawn from the results above: the saved runs and the reference states. Nothing here runs ISSM, so after a change to the presentation, this section alone can be run again. Each figure is also saved as a PDF and a 300-dpi PNG under the name the manuscript gives it (`spinup`, `idealized_results`, `luggye_results` and `summary`). Colour is the sliding law throughout (blue power law, orange effective pressure); line style is the model. Speeds are depth-averaged; in the response figures, the speed is taken 100 m behind the initial terminus (in plan view, the mean within 100 m of the centreline or flowline), and fluxes are per unit width (of the valley, or of Luggye's observed terminus).

In [22]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.tri import Triangulation, LinearTriInterpolator
from matplotlib.patches import Rectangle
from matplotlib import patheffects

colors = {'power-law': '#1f5fa8', 'effective-pressure': '#eb6834'} #blue and orange, which colour-blind readers tell apart (simulated CVD ΔE 23 or more), and dark and light, so they differ in greyscale too
dark_colors = {'power-law': '#0f3566', 'effective-pressure': '#9a3a12'} #darker shades of the same hues, for thin lines drawn on top of thick ones
law_names = {'power-law': 'power law', 'effective-pressure': 'effective pressure'}
calving_names = {'rate law': 'rate law', 'position law': 'position law', 'trivial law': 'fixed front'}
ink, ink_2, grid_ink, ice_color, bed_color, lake_color = '#0b0b0b', '#52514e', '#e4e3df', '#f0f5fb', '#c9c8c3', '#a9cbee'
speed_map = LinearSegmentedColormap.from_list('speed', ['#eef5fd', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
thickness_map = LinearSegmentedColormap.from_list('thickness', ['#f0f6f1', '#a6d1bb', '#5aa585', '#24715a', '#0d3d30']) #a second hue, so the two kinds of map are not confused
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9, 'axes.edgecolor': ink_2, 'axes.labelcolor': ink,
                     'xtick.color': ink_2, 'ytick.color': ink_2, 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'savefig.facecolor': 'white', 'legend.frameon': False, 'pdf.fonttype': 42})

def save_figure(fig, name): #the figure under the name the manuscript gives it: name.pdf, and name.png at 300 dpi
    for extension in ['pdf', 'png']:
        fig.savefig(f'{name}.{extension}', dpi = 300, bbox_inches = 'tight')

def letter(ax, text, title = None): #panel letter, top left, and a title after it
    ax.annotate(text, (0, 1), xycoords = 'axes fraction', xytext = (0, 5), textcoords = 'offset points', fontweight = 'bold', fontsize = 10, va = 'bottom')
    if title:
        ax.annotate(title, (0, 1), xycoords = 'axes fraction', xytext = (13, 5), textcoords = 'offset points', fontsize = 9.5, va = 'bottom')

def tidy(ax): #hairline zero lines and grid
    ax.axhline(0, color = ink_2, linewidth = 0.6, zorder = 1)
    ax.grid(color = grid_ink, linewidth = 0.6)
    ax.set_axisbelow(True)

def draw_profile(ax, p, L, x_max, lines = (), surface = True): #a flowline glacier: bed, ice and lake; lines: more lines (another bed or surface), as (y, plot keywords)
    x, ice = p['x'], p['x'] <= L
    bottom = min([p['bed'].min()] + [np.nanmin(y) for y, _ in lines]) - 20
    ax.fill_between(x, bottom, p['bed'], color = bed_color, linewidth = 0)
    ax.fill_between(x[ice], p['base'][ice], p['surface'][ice], color = ice_color, linewidth = 0)
    lake = (x >= L) & (p['bed'] < 0)
    ax.fill_between(x[lake], p['bed'][lake], 0, color = lake_color, linewidth = 0)
    if surface:
        ax.plot(x[ice], p['surface'][ice], color = ink, linewidth = 0.9)
    ax.plot([x[ice][-1]]*2, [p['base'][ice][-1], p['surface'][ice][-1]], color = ink, linewidth = 0.9) #the ice cliff
    ax.plot(x, p['bed'], color = ink_2, linewidth = 0.8)
    for y, style in lines:
        ax.plot(x, y, **style)
    ax.set_xlim(0, x_max)
    ax.set_ylim(bottom, None)
    ax.set_ylabel('elevation (m)')
    ax.grid(color = grid_ink, linewidth = 0.6)
    ax.set_axisbelow(True)

def draw_speeds(ax, lines, L, x_max): #depth-averaged speed along a flowline: lines as (profile, plot keywords)
    for p, style in lines:
        ice = p['x'] <= L
        ax.plot(p['x'][ice], p['speed'][ice], **style)
    ax.set_xlim(0, x_max)
    ax.set_ylim(0, None)
    ax.set_ylabel('speed (m/a)')
    ax.grid(color = grid_ink, linewidth = 0.6)

def draw_map(ax, x, y, triangles, levelset, speed, v_max, lines = (), clip = None, levels = None, cmap = speed_map): #a field (speed, by default) on the ice of a plan-view mesh, with the front (black) and lines (white)
    #clip: a patch to show only part of it; levels: filled contours between these speeds, instead of continuous shading
    ice = np.max(levelset[triangles], axis = 1) <= 0
    if levels is None:
        image = ax.tripcolor(x, y, triangles[ice], speed, shading = 'gouraud', cmap = cmap, vmin = 0, vmax = v_max, rasterized = True)
    else:
        image = ax.tricontourf(Triangulation(x, y, triangles, mask = ~ice), speed, levels = levels, cmap = cmap)
    ax.tricontour(x, y, triangles, levelset, levels = [0], colors = ink, linewidths = 1.2)
    if clip is not None:
        ax.add_patch(clip)
        clip.set_visible(False)
        image.set_clip_path(clip)
    for line_xy in lines:
        ax.plot(*line_xy, color = 'white', linewidth = 1)
    ax.set_aspect('equal')
    return image

def change(r, key): #changes from the unperturbed run
    y = np.asarray(r[key], dtype = float)
    return y - y[offsets == 0]

def response_grid(curves, title): #rows: calving law; columns: Δ terminus position, Δ speed, Δ flux
    #curves: (label, line style, run, speed key), with run(sliding_law, calving_law) giving the results; the first is drawn on top
    fig, axes = plt.subplots(3, 3, figsize = (10, 8.2), sharex = True)
    columns = [('L', 'Δ terminus position (m)'), ('speed', 'Δ speed (m/a)'), ('flux', 'Δ flux (m²/a)')]
    for i, calving_law in enumerate(calving_laws):
        for j, (column, label) in enumerate(columns):
            ax = axes[i, j]
            if calving_law == 'trivial law' and column == 'L': #a fixed front does not move: the legend goes here
                ax.axis('off')
                continue
            for sliding_law in sliding_laws:
                for n, (name, style, run, speed) in enumerate(curves):
                    style = dict(style)
                    palette, halo, z = style.pop('palette', colors), style.pop('halo', True), style.pop('zorder', 3 + len(curves) - n)
                    effects = [patheffects.Stroke(linewidth = style['linewidth'] + 2, foreground = 'white'), patheffects.Normal()] if halo else None
                    ax.plot(offsets, change(run(sliding_law, calving_law), speed if column == 'speed' else column), color = palette[sliding_law],
                            path_effects = effects, zorder = z, **style)
            tidy(ax)
            ax.axvline(0, color = ink_2, linewidth = 0.6, zorder = 1)
            if i == 0:
                ax.set_title(label)
    for ax in [axes[2, 1], axes[2, 2], axes[1, 0]]:
        ax.set_xlabel('lake-level change (m)')
    axes[1, 0].xaxis.set_tick_params(labelbottom = True)
    for k, ax in enumerate([a for a in axes.ravel() if a.axison]):
        letter(ax, 'abcdefgh'[k])
    handles = [Line2D([], [], color = colors[s], linewidth = 2.5, label = law_names[s]) for s in sliding_laws]
    handles += [Line2D([], [], color = ink if style.get('palette') is dark_colors else ink_2, label = name,
                       **{k: v for k, v in style.items() if k not in ('palette', 'halo', 'zorder')}) for name, style, _, _ in curves]
    axes[2, 0].legend(handles = handles, loc = 'center', fontsize = 9, title = title, title_fontsize = 9.5, alignment = 'left', handlelength = 3)
    fig.tight_layout(rect = (0.03, 0, 1, 1))
    for i, calving_law in enumerate(calving_laws): #row labels
        box = axes[i, 1].get_position()
        fig.text(0.008, (box.y0 + box.y1)/2, calving_names[calving_law], rotation = 90, va = 'center', fontweight = 'bold', fontsize = 10)
    return fig, axes

#line styles of the models: solid, dashed, dotted (round dots), each thick enough to read at a glance
flowline_style = dict(linewidth = 2.2)
thin_style = dict(linewidth = 0.9, palette = dark_colors, halo = False, zorder = 20) #drawn on top of the thick line, in its darker shade
dashed_style = dict(linewidth = 1.6, linestyle = (0, (4, 1.8)))
dotted_style = dict(linewidth = 1.9, linestyle = (0, (0.1, 2)), dash_capstyle = 'round')

## Figure 1: the reference states

A row per glacier: the flowline on the left (geometry over speed), the plan view on the right, where each map is cut along its centreline or flowline into the depth-averaged speed (above) and the ice thickness (below). (a) The idealized flowline glacier, spun up on the retrograde bed (filled) and on the prograde bed (dashed): the surface under power-law and effective-pressure sliding, thick on the retrograde bed and thin on the prograde bed, and (b) the depth-averaged speed, likewise. (c, d) The idealized glacier in plan view, in a valley 500 m wide, spun up under power-law sliding with no-slip walls (c) and free-slip walls (d), over its lowest 1.25 km, in filled contours, with the front in black. (e) Luggye's flowline on the bed inverted for power-law sliding (the bed inverted for effective-pressure sliding dashed), and (f) its depth-averaged speed under either sliding law. (g, h) Luggye in plan view on the flowline's bed under power-law sliding (g) and effective-pressure sliding (h), turned so that its ice flows to the right (north points left), with the front (black) and the flowline (white). In each row, the upper colour bar is the speed and the lower the thickness, for both maps; the scale bar applies to both Luggye maps. Elevations are relative to the lake level.

In [23]:
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.transforms import blended_transform_factory
km = FuncFormatter(lambda v, _: f'{v/1000:g}') #distances on the axes in km

def split_map(ax, x, y, triangles, levelset, cut_x, cut_y, upper, lower):
    #a plan-view map cut along a line running across it from left to right (extended beyond it), with one field above the line and another below:
    #upper and lower as (field, v_max or filled-contour levels, colour map); returns their images
    images = []
    for (field, scale, cmap), edge in [(upper, 1e7), (lower, -1e7)]:
        clip = PolygonPatch(np.c_[np.r_[cut_x, cut_x[-1], cut_x[0]], np.r_[cut_y, edge, edge]], transform = ax.transData)
        levels = scale if np.ndim(scale) else None
        images.append(draw_map(ax, x, y, triangles, levelset, field, scale[-1] if levels is not None else scale, clip = clip, levels = levels, cmap = cmap))
    return images

def place_label(ax, text, x, y, triangles, levelset, field, low, high, inside):
    #write text in black, without a box, where the field's colours are lightest (lowest), over ice and where inside(x, y)
    t = ax.text(0, 0, text, fontsize = 8.5, ha = 'center', va = 'center', color = ink)
    ax.figure.canvas.draw()
    box = t.get_window_extent()
    (x_a, y_a), (x_b, y_b) = ax.transData.inverted().transform([(box.x0, box.y0), (box.x1, box.y1)])
    w, h = 1.2*abs(x_b - x_a), 1.4*abs(y_b - y_a)
    ice = np.max(levelset[triangles], axis = 1) <= 0
    value = LinearTriInterpolator(Triangulation(x, y, triangles, mask = ~ice), (np.asarray(field, dtype = float) - low)/(high - low))
    (x_0, x_1), (y_0, y_1) = ax.get_xlim(), ax.get_ylim()
    best = None
    for c_x in np.linspace(x_0 + w/2, x_1 - w/2, 40):
        for c_y in np.linspace(y_0 + h/2, y_1 - h/2, 16):
            s_x, s_y = [v.ravel() for v in np.meshgrid(np.linspace(c_x - w/2, c_x + w/2, 7), np.linspace(c_y - h/2, c_y + h/2, 3))]
            v = value(s_x, s_y)
            if np.ma.getmaskarray(v).any() or not np.all(inside(s_x, s_y)): #off the ice, or across the cut
                continue
            if best is None or v.max() < best[0] - 1e-6:
                best = (v.max(), c_x, c_y)
    t.set_position(best[1:])

fig = plt.figure(figsize = (10.5, 8.8))
outer = GridSpec(2, 2, figure = fig, width_ratios = [1.25, 1], hspace = 0.26, wspace = 0.14)
grids = [outer[row, 0].subgridspec(2, 1, height_ratios = [1.75, 1], hspace = 0.32) for row in range(2)]
ax_a = fig.add_subplot(grids[0][0]); ax_b = fig.add_subplot(grids[0][1], sharex = ax_a)
maps = outer[0, 1].subgridspec(2, 1, hspace = 0.2)
ax_c = fig.add_subplot(maps[0]); ax_d = fig.add_subplot(maps[1], sharex = ax_c)
ax_e = fig.add_subplot(grids[1][0]); ax_f = fig.add_subplot(grids[1][1], sharex = ax_e)
luggye_maps = outer[1, 1].subgridspec(2, 1, hspace = 0.2) #room for the scale bar between the maps
ax_g = fig.add_subplot(luggye_maps[0]); ax_h = fig.add_subplot(luggye_maps[1], sharex = ax_g, sharey = ax_g)
thick, thin = dict(linewidth = 1.7), dict(linewidth = 0.9)
labels = [] #the speed and thickness labels, placed once the layout is final

#the idealized flowline glacier on both beds: thick on the retrograde bed, thin (darker) on the prograde bed
draw_profile(ax_a, profiles[('power-law', 'retrograde')], L_0, L_0 + 500, surface = False,
             lines = [(profiles[('power-law', 'prograde')]['bed'], dict(color = ink_2, linewidth = 1, linestyle = (0, (4, 2)), label = 'prograde bed'))])
for sliding_law in sliding_laws:
    for direction, style, palette in [('retrograde', thick, colors), ('prograde', thin, dark_colors)]:
        p = profiles[(sliding_law, direction)]
        ice = p['x'] <= L_0
        ax_a.plot(p['x'][ice], p['surface'][ice], color = palette[sliding_law], zorder = 3 if direction == 'prograde' else 2, **style)
draw_speeds(ax_b, [(profiles[(s, d)], dict(color = palette[s], zorder = z, **style)) for s in sliding_laws
                   for d, style, palette, z in [('retrograde', thick, colors, 2), ('prograde', thin, dark_colors, 3)]], L_0, L_0 + 500)
ax_b.xaxis.set_major_locator(MultipleLocator(1000))
ax_b.set_xlabel('distance along the flowline (km)')
ax_a.legend(fontsize = 8.5, loc = 'upper right')
ax_b.legend(handles = [Line2D([], [], color = colors[s], linewidth = 2, label = law_names[s]) for s in sliding_laws] +
                      [Line2D([], [], color = ink_2, label = 'retrograde bed', **thick), Line2D([], [], color = ink, label = 'prograde bed', **thin)],
            fontsize = 8, loc = 'upper left', ncol = 2)

#the valley glacier, with no-slip and with free-slip walls: speed above the centreline, thickness below
valley = valley_mesh(model(), x_nodes_2d, y_nodes_2d)
x_v, y_v = valley.mesh.x, valley.mesh.y
valley_triangles = valley.mesh.elements.astype(int) - 1
x_min = L_0 - 1250
shown = x_v >= x_min
valley_speeds = {wall: np.hypot(*saved_2d[('power-law', wall)]['velocity']) for wall in walls_2d}
valley_H = {wall: np.asarray(saved_2d[('power-law', wall)]['H'], dtype = float) for wall in walls_2d}
v_max = 5*np.ceil(max(s[shown].max() for s in valley_speeds.values())/5)
speed_levels = np.linspace(0, v_max, int(v_max/2.5) + 1 if v_max <= 40 else int(v_max/5) + 1) #bands of 2.5 m/a (5 m/a if the ice is faster)
H_ice = np.concatenate([H[shown & (x_v <= L_0)] for H in valley_H.values()])
thickness_levels = np.arange(20*np.floor(H_ice.min()/20), 20*np.ceil(H_ice.max()/20) + 1, 20) #bands of 20 m
for ax, wall in zip([ax_c, ax_d], walls_2d):
    image_speed, image_H = split_map(ax, x_v, y_v, valley_triangles, x_v - L_0, [x_min - 1e4, L_0 + 1e4], [0, 0],
                                     (valley_speeds[wall], speed_levels, speed_map), (valley_H[wall], thickness_levels, thickness_map))
    ax.plot([x_min, L_0], [0, 0], color = 'white', linewidth = 1)
    ax.set_xlim(x_min, L_0 + 250)
    ax.set_ylim(-W/2, W/2)
    ax.set_yticks([-W/2, 0, W/2])
    ax.xaxis.set_major_locator(MultipleLocator(500))
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.set_ylabel('across (km)')
    ax.text(L_0 + 20, -W/2 + 20, 'lake', fontsize = 8, color = ink_2, va = 'bottom')
    ax.set_anchor('W')
    labels += [(ax, 'speed', x_v, y_v, valley_triangles, x_v - L_0, valley_speeds[wall], 0, v_max, lambda x, y: y > 15),
               (ax, 'thickness', x_v, y_v, valley_triangles, x_v - L_0, valley_H[wall], thickness_levels[0], thickness_levels[-1], lambda x, y: y < -15)]
ax_c.tick_params(labelbottom = False)
ax_d.set_xlabel('distance along the valley (km)')

#Luggye's flowline
p_L = {s: references[s]['profile'] for s in sliding_laws}
draw_profile(ax_e, p_L['power-law'], L_L, L_L + 200, lines = [(p_L['effective-pressure']['bed'], dict(color = colors['effective-pressure'], linewidth = 1.3,
                                                                                                   linestyle = (0, (4, 2)), label = 'bed, effective-pressure sliding'))])
draw_speeds(ax_f, [(p_L[s], dict(color = colors[s], **thick)) for s in sliding_laws], L_L, L_L + 200)
ax_f.xaxis.set_major_locator(MultipleLocator(500))
ax_f.set_xlabel('distance along the flowline (km)')
ax_e.legend(fontsize = 8.5, loc = 'upper right')
for ax in [ax_a, ax_e]:
    ax.tick_params(labelbottom = False)
for ax in [ax_b, ax_f]:
    ax.xaxis.set_major_formatter(km)

#Luggye in plan view under either sliding law, turned a quarter turn anticlockwise so that its ice (flowing south) flows to the right:
#speed above the flowline, thickness below
turn = lambda x, y: (-(np.asarray(y) - np.mean(fields_L2.y)), np.asarray(x) - np.mean(fields_L2.x)) #north then points left
x_g, y_g = turn(fields_L2.x, fields_L2.y)
luggye_triangles = md_L2.mesh.elements.astype(int) - 1
fl_x, fl_y = turn(*np.array(line.coords).T)
end = np.array([fl_x[-1] - fl_x[-2], fl_y[-1] - fl_y[-2]])/np.hypot(fl_x[-1] - fl_x[-2], fl_y[-1] - fl_y[-2])
cut_x, cut_y = np.r_[fl_x[0] - 1e4, fl_x, fl_x[-1] + 1e4*end[0]], np.r_[fl_y[0], fl_y, fl_y[-1] + 1e4*end[1]] #the flowline, continued across the map
luggye_speeds = {s: np.hypot(*saved_L2[('calibration', s)]['velocity']) for s in sliding_laws}
luggye_H = {s: references_L2[s]['H'] for s in sliding_laws}
v_max_L = 10*np.ceil(max(v[fields_L2.ice].max() for v in luggye_speeds.values())/10)
H_max_L = 50*np.ceil(max(v[fields_L2.ice].max() for v in luggye_H.values())/50)
x_0, x_1 = x_g[fields_L2.ice].min() - 15, x_g[fields_L2.ice].max() + 60 #the ice, and a little of the lake beyond its front
y_0, y_1 = y_g[fields_L2.ice].min() - 15, y_g[fields_L2.ice].max() + 15
above = lambda x, y: y > np.interp(x, cut_x, cut_y) + 15
below = lambda x, y: y < np.interp(x, cut_x, cut_y) - 15
for ax, sliding_law in [(ax_g, 'power-law'), (ax_h, 'effective-pressure')]:
    image_speed_L, image_H_L = split_map(ax, x_g, y_g, luggye_triangles, fields_L2.ice_levelset, cut_x, cut_y,
                                         (luggye_speeds[sliding_law], v_max_L, speed_map), (luggye_H[sliding_law], H_max_L, thickness_map))
    ax.plot(fl_x, fl_y, color = 'white', linewidth = 1)
    ax.set_xlim(x_0, x_1); ax.set_ylim(y_0, y_1)
    ax.set_anchor('W')
    ax.set_xticks([]); ax.set_yticks([])
    for side in ['left', 'bottom']:
        ax.spines[side].set_visible(False)
    labels += [(ax, 'speed', x_g, y_g, luggye_triangles, fields_L2.ice_levelset, luggye_speeds[sliding_law], 0, v_max_L, above),
               (ax, 'thickness', x_g, y_g, luggye_triangles, fields_L2.ice_levelset, luggye_H[sliding_law], 0, H_max_L, below)]
#north arrow, pointing left, in the empty top right corner of the upper map; the scale bar between the maps, on the right: 1 km in two halves
ax_g.annotate('', xy = (0.87, 0.9), xytext = (0.98, 0.9), xycoords = 'axes fraction', arrowprops = dict(arrowstyle = '-|>', color = ink, linewidth = 1.2))
ax_g.text(0.855, 0.9, 'N', transform = ax_g.transAxes, ha = 'right', va = 'center', fontsize = 9, fontweight = 'bold')
gap = blended_transform_factory(ax_g.transData, ax_g.transAxes) #x in metres, y in fractions of the upper map's height
for i, face in enumerate([ink, 'white']):
    ax_g.add_patch(Rectangle((x_1 - 1000 + 500*i, -0.115), 500, 0.06, transform = gap, facecolor = face, edgecolor = ink, linewidth = 0.8, clip_on = False))
ax_g.text(x_1 - 1000 - 0.015*(x_1 - x_0), -0.085, '1 km', transform = gap, ha = 'right', va = 'center', fontsize = 8.5)

letter(ax_a, 'a', 'Idealized glacier, flowline'); letter(ax_b, 'b')
letter(ax_c, 'c', 'Idealized glacier, plan view: no-slip walls'); letter(ax_d, 'd', 'free-slip walls')
letter(ax_e, 'e', 'Luggye Glacier, flowline'); letter(ax_f, 'f')
letter(ax_g, 'g', 'Luggye Glacier, plan view: power law'); letter(ax_h, 'h', 'effective pressure')

#the colour bars, beside the maps once their equal aspect has fixed their size: speed beside the upper map of each row, thickness beside the lower,
#all at the same distance from the maps
fig.canvas.draw()
x_bar = max(ax.get_position().x1 for ax in [ax_c, ax_d, ax_g, ax_h]) + 0.012
for ax, image_, ticks, label in [(ax_c, image_speed, speed_levels[::2], 'speed (m/a)'), (ax_d, image_H, thickness_levels[::2], 'thickness (m)'),
                                 (ax_g, image_speed_L, None, 'speed (m/a)'), (ax_h, image_H_L, None, 'thickness (m)')]:
    box = ax.get_position()
    colorbar = fig.colorbar(image_, cax = fig.add_axes([x_bar, box.y0, 0.011, box.height]), ticks = ticks)
    colorbar.ax.tick_params(labelsize = 7.5)
    colorbar.set_label(label, fontsize = 8)
for label in labels:
    place_label(*label)
save_figure(fig, 'spinup')
plt.show()

## Figure 2: the idealized glacier's response to the lake level

Changes after one year relative to the unperturbed run, for each calving law (rows): the flowline on the retrograde bed (thick) and on the prograde bed (thin, in a darker shade), and the plan view, on the retrograde bed, with no-slip walls (dashed) and free-slip walls (dotted). The gap between the thick and thin lines is the effect of the bed slope; the gap between the two plan-view lines, the effect of buttressing by the walls.

In [24]:
valley_run = lambda wall: (lambda sliding_law, calving_law: saved_2d[(sliding_law, wall, calving_law)])
fig, axes = response_grid([('flowline, retrograde bed', flowline_style, lambda s, c: saved[(s, 'retrograde', c)], 'u_100'),
                           ('flowline, prograde bed', thin_style, lambda s, c: saved[(s, 'prograde', c)], 'u_100'),
                           ('plan view, no-slip walls', dashed_style, valley_run('no slip'), 'u_t'),
                           ('plan view, free-slip walls', dotted_style, valley_run('free slip'), 'u_t')],
                          'Idealized glacier')
save_figure(fig, 'idealized_results')
plt.show()

## Figure 3: Luggye's response to the lake level

As Figure 2, for Luggye: the flowline (solid) and the plan view (dashed), on the same bed along the flowline.

In [25]:
fig, axes = response_grid([('flowline', flowline_style, lambda s, c: saved_L[(s, c)], 'u_100'),
                           ('plan view', dashed_style, lambda s, c: saved_L2[(s, c)], 'u_t')],
                          'Luggye Glacier')
save_figure(fig, 'luggye_results')
plt.show()

## Figure 4: all six model glaciers at a glance

The change after one year with the lake 10 m higher (filled triangles pointing up) and 10 m lower (open triangles pointing down), for each model glacier and calving law: the change in terminus position (m), and the changes in speed and flux as percentages of the unperturbed run's. The table below the figure lists the values.

In [26]:
models = [('idealized flowline, retrograde', lambda s, c: saved[(s, 'retrograde', c)], 'u_100'),
          ('idealized flowline, prograde', lambda s, c: saved[(s, 'prograde', c)], 'u_100'),
          ('idealized plan view, no-slip walls', lambda s, c: saved_2d[(s, 'no slip', c)], 'u_t'),
          ('idealized plan view, free-slip walls', lambda s, c: saved_2d[(s, 'free slip', c)], 'u_t'),
          ('Luggye flowline', lambda s, c: saved_L[(s, c)], 'u_100'),
          ('Luggye plan view', lambda s, c: saved_L2[(s, c)], 'u_t')]
def summary(run, speed, offset): #changes at a lake-level offset: terminus position (m), speed and flux (% of the unperturbed run)
    at = lambda key: np.asarray(run[key], dtype = float)
    i, i_0 = int(np.flatnonzero(offsets == offset)[0]), int(np.flatnonzero(offsets == 0)[0])
    return at('L')[i] - at('L')[i_0], 100*(at(speed)[i]/at(speed)[i_0] - 1), 100*(at('flux')[i]/at('flux')[i_0] - 1)

rows = [(calving_law, name, run, speed) for calving_law in calving_laws for name, run, speed in models]
fig, axes = plt.subplots(1, 3, figsize = (10, 7.8), sharey = True)
y = np.arange(len(rows))[::-1] + np.repeat(np.arange(len(calving_laws))[::-1]*0.8, len(models)) #a gap between the calving laws
table = []
for k, (calving_law, name, run, speed) in enumerate(rows):
    for n, sliding_law in enumerate(sliding_laws):
        up, down = summary(run(sliding_law, calving_law), speed, 10), summary(run(sliding_law, calving_law), speed, -10)
        table.append((calving_names[calving_law], name, law_names[sliding_law]) + up + down)
        for j, ax in enumerate(axes):
            if calving_law == 'trivial law' and j == 0:
                continue
            y_k = y[k] + (0.17 if n == 0 else -0.17)
            ax.plot([down[j], up[j]], [y_k, y_k], color = colors[sliding_law], linewidth = 1, alpha = 0.5)
            ax.plot(up[j], y_k, '^', color = colors[sliding_law], markersize = 7.5, markeredgecolor = 'white', markeredgewidth = 0.8, zorder = 4)
            ax.plot(down[j], y_k, 'v', color = 'white', markeredgecolor = colors[sliding_law], markersize = 7, markeredgewidth = 1.4, zorder = 4)
for ax, title in zip(axes, ['Δ terminus position (m)', 'Δ speed (%)', 'Δ flux (%)']):
    ax.axvline(0, color = ink_2, linewidth = 0.6)
    ax.grid(axis = 'x', color = grid_ink, linewidth = 0.6)
    ax.set_axisbelow(True)
    ax.set_title(title)
    ax.tick_params(axis = 'y', length = 0)
axes[0].set_yticks(y, [name for _, name, _, _ in rows])
for k, ax in enumerate(axes):
    letter(ax, 'abc'[k])
handles = [Line2D([], [], color = colors[s], linewidth = 2.5, label = law_names[s]) for s in sliding_laws]
handles += [Line2D([], [], color = ink_2, marker = '^', markersize = 7.5, linewidth = 0, label = 'lake 10 m higher'),
            Line2D([], [], color = ink_2, marker = 'v', markersize = 7, markerfacecolor = 'white', markeredgewidth = 1.4, linewidth = 0, label = 'lake 10 m lower')]
fig.legend(handles = handles, loc = 'lower center', ncol = 4, fontsize = 9, bbox_to_anchor = (0.55, -0.01))
fig.tight_layout(rect = (0, 0.04, 1, 1))
fig.canvas.draw()
left = min(t.get_window_extent().x0 for t in axes[0].get_yticklabels()) #the group labels line up with the longest model name
left = axes[0].transAxes.inverted().transform((left, 0))[0]
for g, calving_law in enumerate(calving_laws):
    axes[0].text(left, y[g*len(models)] + 0.6, calving_names[calving_law], transform = axes[0].get_yaxis_transform(), fontweight = 'bold', fontsize = 9.5)
save_figure(fig, 'summary')
plt.show()

print(f"{'':12s} {'model glacier':36s} {'sliding law':19s} {'lake 10 m higher: ΔL (m), Δu (%), Δq (%)':>42s}   {'lake 10 m lower':>22s}")
for calving_law, name, sliding_law, *values in table:
    print(f'{calving_law:12s} {name:36s} {sliding_law:19s} {values[0]:+16.1f} {values[1]:+8.1f} {values[2]:+8.1f}       {values[3]:+8.1f} {values[4]:+7.1f} {values[5]:+7.1f}')

             model glacier                        sliding law           lake 10 m higher: ΔL (m), Δu (%), Δq (%)          lake 10 m lower
rate law     idealized flowline, retrograde       power law                       -7.8    -16.0    -22.0           +8.1   +11.6   +17.3
rate law     idealized flowline, retrograde       effective pressure              -4.6     +6.0     -2.8           +5.7    -0.9    +6.3
rate law     idealized flowline, prograde         power law                       -7.6    -14.7    -20.5           +7.9   +10.7   +16.4
rate law     idealized flowline, prograde         effective pressure              -4.2     +6.7     -0.5           +5.4    -0.1    +6.2
rate law     idealized plan view, no-slip walls   power law                       -6.2    -12.5    -19.8           +6.4   +10.7   +16.8
rate law     idealized plan view, no-slip walls   effective pressure              -4.8     +0.8     -6.7           +5.0    +1.4    +7.4
rate law     idealized plan view, free-slip wa

# Notes on the method

## Layers, calibration and perturbation runs

**Vertical layers.** On the same glacier, terminus speeds with 5 layers were 3–6% lower than with 20, and with 10 they are within 1%.

**Calibration.** `calibrate` finds the value of $k$ or $f^*$ for which the unperturbed run's terminus drift (`drift`, in metres over the year) is within 0.5 m of zero. More calving means more retreat, so the drift decreases as either value grows. After `guess`, the second try is one `step` towards zero drift (more calving if the terminus advanced, less if it retreated); each later try is where the straight line through the last two tries reaches zero drift. Once tries lie on both sides of zero drift, the line is drawn between the closest on either side, and a side that has survived two tries in a row counts half, so that the next try moves away from it (the drift can change abruptly, as calving removes ice in discrete pieces). It stops after 12 tries, or when the closest tries on either side differ by less than `step`/50, and returns the try with the smallest drift.

**Flowline runs.** A perturbation run ramps the lake level to its offset and advances the terminus position $L$ by the calving law, as in the icepack version. The ISSM front sits on the node nearest to $L$: with a grounded front inside an element, ISSM largely loses the acceleration of the ice toward the front, so the meshes have 1 m nodes where the front moves. Ice gained by an advance takes the thickness of the old front. After the last step, the velocity is solved once more with the front on its final node; the terminus speed is taken there, and the flux is the depth-averaged speed times the thickness at the grounding line.

**Plan-view runs.** The front moves with ISSM's level set under `rate_law` (at $k$ times the local water depth) or `position_law` (where $f_{af} < f^*$), or stays in place. Where the front runs through elements, `issm_helper` solves the stress balance with the front on the vertices just behind and just ahead of it and interpolates between the two (see `diagnostic_solve`), so the velocity changes smoothly as the front moves; the length, flux and speed along the front from `front_diagnostics` are interpolated the same way.

**The speed near the terminus.** In plan view, the speed is taken at a fixed place, `speed_distance` (100 m) behind the initial front on lines down the glacier within 100 m of the centreline (or of the flowline), and averaged over them, as a velocity map would see it. The speed of the front itself jumps from one lake level to the next as the position law cuts the front back node by node. In a test on the idealized glacier in plan view (power law, no-slip walls, position law, lake 0 to 8 m higher), the speed of the front rose by 3.3 m/a, with a scatter of 0.33 m/a about a smooth curve and two reversals; 100 m behind the initial front, the speed rose by 4.1 m/a, with a scatter of 0.17 m/a and none. A speed taken a fixed distance behind the moving front barely changed (by −0.03 m/a at 100 m): the speed-up is the ice there coming closer to the retreating front. The flowlines record the speed at the same place, for the figures.

## Idealized glacier in plan view

**Solver and running time.** The plan-view models need ISSM built with MUMPS (on a single process without MUMPS, `issm_helper` falls back to PETSc's serial direct solver, which works but is slow). With 8 processes on Snellius, the four spin-ups (up to 1000 years each) and calibrations took about 3 hours, and the lake-level runs about 6.

**Mesh.** The vertices are 5 m apart along the valley where the front can move, 100 m apart far upstream, and across the valley from 50 m in the middle to 10 m at the walls, where the ice shears most; with 15 rows across, the flux at the front is within 1% of that with 31.

## Luggye in plan view

**Inflow and closed edges.** Ice enters across about 480 m of the top edge of the DEM, where the consensus thickness counts it as glacier, at 10 m/a in the middle, falling to zero at either end as a parabola. The rest of the top edge is a flat bench outside the consensus glacier, and the domain also meets the edge of the DEM where a small tributary joins from the east, through a notch about 150 m long. Both are closed: ice pushed into them piles up (on the bench it barely flows, in the notch it has nowhere to go), and a bed inversion in plan view runs away there. With 8 processes on Snellius, the calibration takes about 1 hour and the lake-level runs about 6.

**A bed inverted in plan view.** Inverted in the same way as the flowline's, the plan-view bed gives no usable glacier. Inverted for a steady glacier, it converges with the terminus in about 5 m of water (power law): with no surface mass balance, a glacier in balance discharges at its front only what enters across the top of the DEM, about 390 m²/a per metre of terminus, and the observed terminus, 34 m above the lake, carries that little only when it is about 40 m thick. Inverted for a thinning glacier (uniform thinning up to the observed 4.67 m/a, or thinning falling with height from the 9.1 m/a observed lowest down to 1.3 m/a), it does not settle. With no surface mass balance, the ice can thin only by flowing, and the slow ice along the valley sides and where the steep slope meets the overdeepened tongue cannot thin at the target rate, while the lowest 300 m of the tongue thin at 8–17 m/a when the bed under them is deep. The inversion keeps lowering the bed where the ice thins too little and raises it under the tongue into a sill, with the terminus heading for 5–20 m of water.